# <center>ClaudeCode 第一节课：AI 编程工作台搭建与配置</center>

&emsp;&emsp;今天这节课只解决一个非常具体的问题：把后续两节课要使用的 ClaudeCode AI 编程工作台搭起来。它不是一节抽象方法论课，也不是一节 Agent 原理课，而是一节基础设施课。你需要在本地拥有一个能启动、能接入模型、能证明路由正确、能读取项目规则、能遵守权限边界的工作台。

&emsp;&emsp;这件事看起来像安装软件，实际上决定了后续学习能不能顺利推进。没有稳定的 ClaudeCode 环境，后面很难做项目理解；没有 DeepSeek 路由证据，后面很难判断模型调用是否真实生效；没有 `CLAUDE.md`、settings 和项目命令，后面每次协作都会重新解释规则，学习成本会被反复浪费。

&emsp;&emsp;本课采用"搭积木"的路径展开：我们先把 ClaudeCode 客户端跑起来（支持 CLI 和 VS Code 扩展两条等效路径），再接入 DeepSeek，然后用对比实验确认配置真的生效，接着深入开源路线把 cc-haha（一款开源的 ClaudeCode-like CLI，基于泄露源码做学习用） + CCR（Claude Code Router 的简称，本机跑的协议路由层，把 Anthropic 格式的请求转发给任意 OpenAI 兼容或 Anthropic 兼容后端） + DeepSeek 完整链路也跑通，最后用 6 条立刻可用的提示词把工作台用起来。整个过程贯穿两条主线：一条显性线是把工作台搭起来；一条隐性线是训练"用外部证据判断模型路由"的协作底层观念——**能对话不等于能验证**。

> 📌 **目标受众与前置要求**：本课面向希望用 ClaudeCode / Codex / Cursor 完成真实项目协作的开发者。双路径并行覆盖：路径 A（终端 CLI）需要能打开终端 + 了解 API Key 安全要求；路径 B（VS Code 扩展）只需安装 VS Code。两条路径均不预设 Python/ML 基础。

> 📌 **学完本节你将带走 8 件产物**：① 可启动的 ClaudeCode 工作台（CLI 或 VS Code 扩展）；② DeepSeek 模型接入 + 路由证据；③ 全局与项目级 CLAUDE.md；④ .claude/settings.json 权限配置；⑤ 两个项目命令；⑥ 6 条立刻可用的提示词模板；⑦ 可双脚本启动的 cc-haha + CCR + DeepSeek 完整开源链路；⑧ 架构笔记模板（第二节课"项目初读"的记录容器）。

> 📅 **时效性说明**：本课全部源码与命令引用截止 2026 年 5 月初，涉及 ClaudeCode 安装方式、cc-switch v3.14.1、DeepSeek 模型名（`deepseek-v4-pro` / `deepseek-v4-flash`）、VS Code 扩展、settings 权限语法、CCR v2.0.0 配置字段都随版本飘移。每个版本敏感章节开头有 📅 时效性说明。遇到命令或字段不一致，以官方文档为准。

> 📖 **核心术语速查**（首次接触读者必读，已熟悉可跳过）：
>
> - **ClaudeCode**：Anthropic 官方的 CLI/IDE 编程助手（本课主角）
> - **cc-switch**：ClaudeCode 多 provider 配置管理桌面工具（GUI），见第 4 章
> - **provider**：模型服务商（"提供方"），本课主用 DeepSeek；cc-switch 内以 provider 配置形式管理多个服务商
> - **CCR**：Claude Code Router，本机协议路由层，把 Anthropic 请求转发给 OpenAI 或 Anthropic 兼容后端，见第 6 章
> - **cc-haha**：开源的 ClaudeCode-like CLI（基于泄露源码做学习参考），见第 6 章
> - **transformer**：CCR 配置中负责协议格式转换的字段（如 `"Anthropic"` / `"deepseek"`），大小写敏感，见 6.10 节
> - **Skill / Subagent**：ClaudeCode 的高阶能力（第三节课展开，本课仅 7.6 节预热），分别对应"可复用任务流"和"专项子智能体"
> - **路径 A / 路径 B**：本课的两条等效安装路径，A=终端 CLI，B=VS Code 扩展（决策框见下方）

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 0-0 工具版本基线表</font></p>
<div class="center">

| 工具 | 版本 | 验证命令 / 入口 | 说明 |
|---|---|---|---|
| ClaudeCode | `2.1.142` | `claude --version` | 第 3 章安装 |
| cc-switch | `v3.14.1`（2026-04-23 发布） | 桌面 GUI 应用「关于」 | 第 4 章使用 |
| CCR | `v2.0.0` | `ccr --version` | 第 6 章使用 |
| Node.js | `20+ LTS`（推荐 v24 Krypton） | `node --version` | 第 2 章安装；Node 18 已 EOL |
| Bun | 最新稳定版 | `bun --version` | 第 6.7 节安装 |
| DeepSeek 模型 | `deepseek-v4-pro` / `deepseek-v4-flash` | cc-switch 模型字段 / CCR config | 旧 ID `chat`/`reasoner`/`v3.2` 于 **2026-07-24 废弃** |
| 基线日期 | 2026-05-15 | — | 实测日期 |

</div>

&emsp;&emsp;如果你看到的工具版本和上表不一致，**优先以你机器上 `--version` 实测为准**；本课流程在 minor 版本差异下大部分可直接复用，重要差异会在各章 📅 时效性说明里单独标出。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 0-1 本课章节总览</font></p>
<div class="center">

| 章节 | 主题 | 类型 | 是否必做 |
|---|---|---|---|
| 第一章 | 工作台地图 + 教学双线宣告（**本课件内部教学框架**，详见 1.1 节）| 认知 | 必做 |
| 第二章 | 环境预检与安全边界 | 动手 | 必做 |
| 第三章 | 安装 ClaudeCode（CLI + VS Code 双路径） | 动手 | 必做 |
| 第四章 | 接入 DeepSeek（cc-switch GUI 主路径 + CLI 补充） | 动手 | 必做 |
| 第五章 | 四组对比验证（**本课件内部教学框架**，详见第 5 章）| 验证 | 必做 |
| 第六章 | 开源路线深度部署（cc-haha + CCR + DeepSeek） | 动手 | 必做（核心） |
| 第七章 | 部署后立刻可用的提示词 | 应用 | 必做 |
| 第八章 | 总验收 + 课后任务 | 验证 | 必做 |

</div>

> 🛣️ **路径 A vs 路径 B 决策框**（动手前先看，避免走错路）：
>
> - **走路径 A（终端 CLI）**：日常用终端 / 想看完整命令日志 / 需要 SSH 远程开发 → 完整跟第 2 章 + 3.1-3.4 节 + 第 4 章 cc-switch GUI（仍推荐用 GUI 配 provider，节省手写 settings 工作量）
> - **走路径 B（VS Code 扩展）**：日常以 VS Code 为主 / 不习惯终端 / 初次接触 AI 编程 → 第 2 章 2.1-2.5 节先跳过 → 3.5 节装扩展 → 第 4 章用 cc-switch GUI → ⚠️ **第六章必做核心：必须回头补 2.1-2.5 节（Node）+ 6.7 节（Bun）**（cc-haha + CCR 链路硬性依赖 Node 20+ 和 Bun）
> - **不确定 / 想都体验**：默认走路径 A，第 4 章用 cc-switch GUI 体验 GUI 流程；⚠️ **第六章对所有学员同样必做**（无论 A/B 路径都依赖 Node 20+ 和 Bun），路径 A 学员第二章已经装好，路径 B 学员仍需要按"路径 B"行的提示在第六章前补做

---

## <center>第一章：为什么第一节课只搭工作台</center>

&emsp;&emsp;本章我们先把本课的终点说清楚。后续两节课要进入项目理解、任务拆解和真实修改，如果第一节课没有把工具链准备好，后面所有练习都会被安装、认证、权限和路由问题打断。

&emsp;&emsp;这一章不急着输入命令，而是先建立工作台地图：哪些文件是必须产物，哪些工具是主线，哪些内容只是研究路线。你会看到本课不是简单装一个 CLI，而是搭出一套可验证的 AI 协作基础设施。

&emsp;&emsp;本章的核心判断标准很简单：学完之后，你应该能说清楚自己机器上有哪些配置、每个配置负责什么、哪些内容会影响后续课程。

&emsp;&emsp;本节课只解决一个问题：让后续两节课有一个可以稳定使用、可以验证、可以对比的 ClaudeCode 工作台。完成后，你的环境中应该包含以下产出物：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 1-1 本节课工作台产出物地图</font></p>
<div class="center">

| 产出物 | 位置或入口 | 用途 | 是否必做 |
|---|---|---|---|
| ClaudeCode 客户端 | `claude` 命令 或 VS Code 扩展 | 后续主线开发与项目协作 | 必做 |
| DeepSeek 模型接入 | cc-switch 或等价环境变量 | 让 ClaudeCode 调用 DeepSeek | 必做 |
| 路由验证记录 | DeepSeek 控制台、cc-switch 状态、配置截图或日志 | 证明请求确实走到目标模型 | 必做 |
| 全局规则文件 | `~/.claude/CLAUDE.md` | 保存个人所有项目通用的协作偏好 | 必做 |
| 项目规则文件 | `CLAUDE.md` 或 `.claude/CLAUDE.md` | 让模型理解当前项目背景、协作方式和项目约束 | 必做 |
| 项目本地规则 | `CLAUDE.local.md` | 保存当前项目的个人偏好，不提交仓库 | 选做 |
| 权限配置 | `.claude/settings.json` | 限制敏感文件和高风险命令 | 必做 |
| 项目命令 | `.claude/commands/*.md` | 把高频任务固化成可复用入口 | 必做 |
| 架构笔记模板 | `ARCHITECTURE_NOTES.md` | 为第二节课的项目理解做铺垫 | 必做 |
| 开源完整链路 | cc-haha + CCR + DeepSeek + 启动脚本 | 深入理解 ClaudeCode 类产品的协议链路 | 必做（第 6 章） |

</div>

&emsp;&emsp;本课的主线是官方 ClaudeCode。开源路线（cc-haha + CCR）作为深度部署训练保留——它帮助你理解工具链结构，并且在第六章是必做内容。

### 1.1 教学双线宣告

&emsp;&emsp;在进入第二章之前，我们要先把贯穿整个第一节课的"两条主线"明确告知你（**"教学双线"是本课件内部教学框架，非业界通用术语**——它是把"工具搭建"和"协作直觉训练"显式拆出来的一种教学组织方式，方便你在每章末尾自检两条线各推进了多少）。这两条线会从第二章一直贯穿到第八章，每个章节都会同时推进它们。

&emsp;&emsp;**显性线**是我们这节课最直观的目标：把 ClaudeCode + DeepSeek 工作台搭起来。从 Node.js 环境预检开始，一路到 ClaudeCode 客户端安装、cc-switch 配置 provider、官方主线对话、开源路线完整链路，最后落到 CLAUDE.md / settings / commands 和 6 条立刻可用的提示词。这条线学完后，你应该能直接进入第二节课的项目协作。

&emsp;&emsp;**隐性线**是我们这节课更重要的训练：学会用外部证据判断模型路由。这是一个底层观念——**能对话不等于能验证**。模型回答你"我是 DeepSeek"不是路由证据，因为它可能根据提示词猜测，也可能被错误路由后仍然能正常回复。真正的路由证据我们要在三个地方找：cc-switch 当前 provider 状态、DeepSeek 控制台的调用记录、以及第六章会重点讲的"三处证据凑齐"（本课件内部教学框架，非业界通用术语）方法。

&emsp;&emsp;两条线的优先级很清楚：**隐性线比显性线更重要**。如果你只完成了显性线（工作台跑起来），但没建立"用外部证据验证路由"的习惯，后续两节课你仍然会反复掉入"模型说什么我信什么"的陷阱。本课我们设计的所有对比实验、所有验证记录表、所有"路由证据"小节，目的都是把隐性线烙进你的协作直觉里。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180023305.png" width=85%></div>

<p align="center"><font face="黑体" size=3>图 1-1 本课双线并进时间线——显性线（工具搭建）与隐性线（协作观念训练）并行推进，隐性线更重要</font></p>

---

## <center>第二章：预检环境与安全边界</center>

&emsp;&emsp;明确了工作台地图之后，我们下一步是确认机器环境不会在关键环节掉链子。预检不是形式流程，而是把最常见的失败点提前暴露出来。

&emsp;&emsp;本章我们会检查终端、Git、Node.js、npm、网络、DeepSeek API Key 和实验目录。所有检查都围绕一个目标：让后续安装和模型接入有清楚的前置条件。同时，我们会在本章一次性立住安全边界——这是后续所有 AI 工程协作的底线，后面章节统一引用本章定义，不再重复展开。

> 💡 **双路径分流提示**：如果你计划全程使用 VS Code 扩展路径（路径 B），可以**跳过本章 2.1–2.5 节的 Node.js / npm 安装环节**，直接看本章开头的「全局安全边界」blockquote（紧跟在本提示下方），然后进入第三章第 3.5 节。VS Code 扩展本身不依赖本机 Node.js。
>
> ⚠️ **路径 B 学员重要提醒**：第六章「开源路线深度部署」在本课中是**必做核心**（不能跳过），而 cc-haha + CCR 链路硬性依赖 Node 20+ LTS 和 Bun。建议你此刻就**记下一件事**：跳到第 3.5 节之前先在便签上写"第六章前补做 Node 20+ 安装"。到第六章开场（6.0 节）会再次提示你回头补做 2.1–2.5 节（Node）和 6.7 节（Bun），届时按提示回补即可。

> ⚠️ **全局安全边界（本课所有章节统一遵守）**：
>
> 1. **不读取 `.env` / `.env.*` / `secrets/**` / `config/credentials*` 等敏感文件**——无论是手写命令还是 AI 代答，都不应该把这些文件内容打开
> 2. **不把真实 API Key、Token、账号密码写入会被提交到仓库的文件**——包括 `CLAUDE.md`、`.claude/settings.json`、截图、聊天记录
> 3. **真实 Key 只放三个地方**：cc-switch 的本地配置数据库、`.claude/settings.local.json`（已加入 `.gitignore`）、或 shell profile（`.zshrc` / `.bashrc`）
> 4. **后续章节遇到"安全边界"字样时，统一指向本块定义**，不再重复展开

&emsp;&emsp;我们先确认你的机器具备以下条件。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 2-1 环境预检清单</font></p>
<div class="center">

| 项目 | 自检命令或检查方式 | 通过标准 |
|---|---|---|
| 终端可用 | 打开 Terminal、PowerShell 或 WSL | 能执行基本命令 |
| Git | `git --version` | 输出版本号 |
| Node.js | `node -v` | **Node 20+ LTS（当前 LTS 为 Node 24 Krypton）** |
| npm | `npm -v` | 输出版本号 |
| 网络 | 能访问官方文档、GitHub、DeepSeek 控制台 | 页面可打开 |
| DeepSeek API Key | DeepSeek 控制台 `https://platform.deepseek.com/api_keys` | 已创建可用 Key |
| 项目目录 | 任意空目录 | 后续能创建文件 |

</div>

> 📅 **时效性说明**：Node 18 已于 2025 年 4 月 30 日 EOL，本课统一推荐 Node 20+ LTS。第六章的 Claude Code Router（CCR v2.0.0）引擎要求 Node 20+，如果还在用 Node 18 会直接启动失败。

### 2.1 Node.js 与 npm 安装范式

&emsp;&emsp;ClaudeCode 的 npm 备用安装、cc-switch 相关依赖、CCR 安装以及部分开源工具链都会用到 Node.js 和 npm。这里先把安装范式讲清楚：**Node.js 是 JavaScript 运行时，npm 是随 Node.js 一起使用的包管理器**。大多数情况下，你不需要单独安装 npm；安装 Node.js 后再验证 `npm -v` 即可。

&emsp;&emsp;本课的目标是先把 ClaudeCode 工作台稳定搭起来，因此 macOS 默认使用 Homebrew 安装 Node.js：路径直观、命令少、和多数 macOS 开发工具链一致。如果你后续需要在多个项目之间切换 Node 版本，再使用 `nvm` 作为进阶备选。两种方式不要混着抢同一个 `node` 命令，否则容易出现"明明切换了版本，`node -v` 还是旧版本"的 PATH 问题。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 2-2 Node.js 与 npm 安装路线选择</font></p>
<div class="center">

| 操作系统 | 推荐路线 | 适合场景 | 说明 |
|---|---|---|---|
| macOS | Homebrew | 本课默认路线，只需要稳定安装一个 Node 20+ LTS | 简单、路径直观，适合第一节课快速搭环境 |
| macOS | `nvm` | 需要同时维护多个 Node 版本、按项目切换版本 | 灵活，但依赖 shell 初始化和 PATH 顺序 |
| macOS | Node.js 官网 `.pkg` 安装器 | 无法使用 Homebrew 时 | 简单，但后续切换版本不方便 |
| Windows 原生 | Node.js 官网 `.msi` 安装器 | 本课默认路线，只需要稳定安装一个 Node 20+ LTS | 图形化安装，适合第一节课快速搭环境 |
| Windows 原生 | `nvm-windows` | 希望能切换 Node 版本、后续做多个项目 | PowerShell 中管理版本，Node 与 npm 随版本切换 |
| Windows WSL | `nvm` | 使用 Linux/WSL 终端学习 | 和 macOS/Linux 范式一致 |

</div>

> **常见误解**：`npm` 不是另一个需要单独下载安装的软件。通常安装 Node.js 后，npm 会一起可用。你真正需要确认的是：`node -v` 和 `npm -v` 是否都能输出版本号，以及它们是否来自你期望的安装路径。

### 2.2 macOS 安装 Node.js 与 npm

&emsp;&emsp;macOS 本课默认使用 Homebrew 安装 Node.js。Homebrew 安装后的 `node` 和 `npm` 通常位于 `/opt/homebrew/bin`（Apple Silicon）或 `/usr/local/bin`（Intel Mac），路径清楚，适合课程环境快速验证。

&emsp;&emsp;**步骤一：确认命令行工具可用**

&emsp;&emsp;如果你的机器从未安装过 Xcode Command Line Tools，可以先执行：

In [ ]:
!xcode-select --install

&emsp;&emsp;如果系统提示已经安装，可以继续下一步。

&emsp;&emsp;**步骤二：确认 Homebrew 可用**

In [ ]:
!brew --version

&emsp;&emsp;如果能输出 Homebrew 版本号，继续下一步。如果提示 `brew: command not found`，先到 Homebrew 官网安装：

```text
https://brew.sh/
```

&emsp;&emsp;**步骤三：安装 Node.js LTS 版本**

&emsp;&emsp;本课推荐 Node 20+ LTS，当前课程基线使用 Node 24 Krypton。Homebrew 中可以直接安装 `node@24`：

In [ ]:
!brew install node@24

&emsp;&emsp;如果 Homebrew 提示已经安装但命令不可用，执行链接命令：

In [ ]:
!brew link --overwrite --force node@24

&emsp;&emsp;**步骤四：验证 Node.js 与 npm**

In [ ]:
!node -v
!npm -v
!which node
!which npm

&emsp;&emsp;通过标准：

- `node -v` 输出版本号，要求 **Node 20+ LTS（推荐 v24.x Krypton）**。

- `npm -v` 输出版本号。

- `which node` 通常指向 `/opt/homebrew/bin/node` 或 `/usr/local/bin/node`。

- 新开一个终端后，`node -v` 仍然可用。

&emsp;&emsp;**nvm 备选方案：需要多版本切换时再使用**

&emsp;&emsp;如果你后续需要为不同项目切换 Node 版本，可以改用 `nvm`。注意：如果机器上已经有 Homebrew Node，nvm 切换不等于删除 Homebrew Node；最终执行哪个版本，取决于 `PATH` 里哪个路径排在前面。

&emsp;&emsp;安装 nvm：

In [ ]:
!curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.40.4/install.sh | bash

&emsp;&emsp;安装完成后，重启终端，或按你使用的 shell 重新加载配置：

In [ ]:
!source ~/.zshrc

&emsp;&emsp;验证 nvm 是否可用：

In [ ]:
!command -v nvm

&emsp;&emsp;安装并切换到 LTS：

In [ ]:
!nvm install --lts
!nvm use --lts
!nvm alias default 'lts/*'

&emsp;&emsp;再次验证：

In [ ]:
!node -v
!npm -v
!which node
!which npm

&emsp;&emsp;如果使用 nvm，`which node` 应该指向 `~/.nvm/versions/node/...`。如果仍然指向 `/opt/homebrew/bin/node` 或 `/usr/local/bin/node`，说明当前 shell 还在使用 Homebrew Node，先执行：

In [ ]:
!nvm use --lts
!hash -r

&emsp;&emsp;如果 `command -v nvm` 没有输出，优先检查 `~/.zshrc` 是否存在 nvm 加载片段，或重开终端再试。

### 2.3 Windows 安装 Node.js 与 npm

&emsp;&emsp;Windows 原生环境本课默认使用 Node.js 官网 `.msi` 安装器。它是图形化安装流程，路径稳定，适合第一节课快速完成环境准备。`nvm-windows` 作为备选方案，适合后续需要同时维护多个 Node 版本的学员。

&emsp;&emsp;**步骤一：打开 Node.js 官网下载页**

```text
https://nodejs.org/en/download
```

&emsp;&emsp;选择 **Windows Installer (.msi)**，并确认版本类型是 **LTS**。不要为了"最新"选择 Current 版本；本课要求 Node 20+ LTS，当前课程推荐 Node 24 LTS。

&emsp;&emsp;**步骤二：运行安装器**

&emsp;&emsp;双击下载好的 `.msi` 文件，按安装向导一路继续。没有特殊需求时，使用默认安装目录和默认选项即可。安装完成后，关闭并重新打开 PowerShell，确保 PATH 刷新。

&emsp;&emsp;**步骤三：验证 Node.js 与 npm**

```powershell
node -v
npm -v
where node
where npm
```

&emsp;&emsp;通过标准：

- `node -v` 输出版本号，要求 Node 20+ LTS（推荐 v24.x）。

- `npm -v` 输出版本号。

- `where node` 能看到 Node.js 安装路径，且没有旧版本路径排在前面。

- 新开一个 PowerShell 后，`node -v` 仍然可用。

#### **2.3.1 nvm-windows 备选方案：需要多版本切换时再使用**



&emsp;&emsp;如果你后续需要在 Windows 原生环境中切换多个 Node 版本，可以使用 `nvm-windows`。它和 macOS/Linux 的 `nvm` 不是同一个项目，但目标类似：让你在 Windows 上安装、切换和管理多个 Node.js 版本。

&emsp;&emsp;打开 nvm-windows 的 GitHub Releases 页面：

```text
https://github.com/coreybutler/nvm-windows/releases
```

&emsp;&emsp;下载最新 release 中的 `nvm-setup.zip`，解压后运行 `nvm-setup.exe`。安装完成后，打开 PowerShell 查看可安装版本：

```powershell
nvm list available
```

&emsp;&emsp;选择 **LTS** 版本安装，不要选择 Current 版本。下面用 `<version>` 表示实际版本号：

```powershell
nvm install <version>
nvm use <version>
```

&emsp;&emsp;使用 nvm-windows 后也要验证：

```powershell
node -v
npm -v
where node
where npm
```

&emsp;&emsp;通过标准：

- `node -v` 输出版本号，要求 Node 20+ LTS。

- `npm -v` 输出版本号。

- `nvm ls` 能看到当前版本前有 `*` 标记。

- `where node` 指向 nvm-windows 管理的 Node 路径，而不是旧版 Node.js 安装器残留路径。

&emsp;&emsp;如果你之前已经通过 Node.js 官网 `.msi` 安装过 Node，又决定改用 nvm-windows，建议先卸载旧版本。否则 `where node` 可能出现多个路径，导致你以为切换了版本，但 PowerShell 实际调用的还是旧路径。

### 2.4 官网安装器备用方案

&emsp;&emsp;如果你无法使用版本管理器，可以直接使用 Node.js 官网安装器。入口如下：

```text
https://nodejs.org/en/download
```

&emsp;&emsp;选择 **LTS** 版本，不要为了"最新"选择 Current 版本。LTS 更适合课程、项目和工具链长期使用。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 2-3 Node.js 官网安装器选择</font></p>
<div class="center">

| 系统 | 安装包 | 安装后验证 |
|---|---|---|
| macOS | `.pkg` 安装器 | `node -v`、`npm -v`、`which node` |
| Windows | `.msi` 安装器 | `node -v`、`npm -v`、`where node` |

</div>

&emsp;&emsp;官网安装器的优点是简单，缺点是后续切换版本不方便，并且全局 npm 包权限问题更常见。如果你后续要长期做 AI 工程课程，还是建议使用版本管理器范式。

### 2.5 Node.js 与 npm 安装故障排查

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 2-4 Node.js 与 npm 安装常见问题</font></p>
<div class="center">

| 现象 | 可能原因 | 处理方式 |
|---|---|---|
| `node` 命令不存在 | PATH 未刷新、终端未重启 | 重开终端，检查 `which node` 或 `where node` |
| `npm` 命令不存在 | Node 安装不完整或 PATH 指向异常 | 重新安装 Node LTS，检查 Node 路径 |
| Node 版本低于 20 | 使用了旧版 Node 或 PATH 指向旧路径 | macOS 默认执行 `brew install node@24 && brew link --overwrite --force node@24` |
| macOS `brew: command not found` | 未安装 Homebrew | 先到 `https://brew.sh/` 安装 Homebrew |
| macOS `brew install node@24` 后 `node` 仍不可用 | Homebrew 未链接到 PATH | 执行 `brew link --overwrite --force node@24`，然后重开终端 |
| macOS `nvm: command not found` | shell 配置未加载 | 重开终端，或 `source ~/.zshrc`（zsh）/ `source ~/.bashrc`（bash）|
| macOS 安装 nvm 失败 | 缺少 Xcode Command Line Tools | 运行 `xcode-select --install` |
| macOS `nvm alias default` 后仍显示 Homebrew Node | `/opt/homebrew/bin` 或 `/usr/local/bin` 在 PATH 中优先 | 执行 `nvm use --lts && hash -r`，或统一选择 Homebrew 路线避免混用 |
| Windows `nvm` 可用但 `node` 不变 | 旧 Node 路径优先级更高 | 用 `where node` 排查并卸载旧版本 |
| 全局 npm 安装权限错误 | 安装路径或权限配置异常 | macOS 优先使用 Homebrew；Windows 先重装官方 `.msi` LTS，需要多版本时再用 nvm-windows |

</div>

### 2.6 实验目录与 API Key 准备

&emsp;&emsp;我们推荐准备一个专门目录，不要直接在生产项目里做第一轮实验。

In [ ]:
!mkdir -p ~/claude-code-demo
!cd ~/claude-code-demo

&emsp;&emsp;本课涉及 DeepSeek API Key。请按本章顶部的"全局安全边界"管理：真实 Key 不进 `CLAUDE.md`、不进 `.claude/settings.json`、不进截图、不进公开仓库。

---

## <center>第三章：安装 ClaudeCode（CLI + VS Code 双路径）</center>

&emsp;&emsp;完成环境预检后，我们开始安装 ClaudeCode 客户端。这一章给你两条等效路径：路径 A 是终端 CLI 安装（3.1–3.4 节），路径 B 是 VS Code 扩展安装（3.5 节）。两条路径在第 4 章之后完全等效，可以任选一条。

&emsp;&emsp;本章我们按"理解边界 → 官方安装 → npm 备用 → 常见问题 → VS Code 扩展"的顺序展开。你不会只复制安装命令，还会知道 ClaudeCode、模型服务、认证和端点分别处于哪一层。

&emsp;&emsp;本章结束时，我们的最低验收标准是：CLI 路径下 `claude --version` 能输出版本号（如 `2.1.142`）且 `claude doctor` 能正常运行（输出多行 `✓`/`OK` 检查项，**无 `✗`/`Error` 红色阻断**）；或 VS Code 扩展路径下侧边栏能打开并完成登录验证。只有客户端健康，后面的模型接入才有意义。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 3-1 ClaudeCode 安装阶段的常见误解</font></p>
<div class="center">

| 常见误解 | 真实机制 | 验证方式 |
|---|---|---|
| ClaudeCode 本身就是模型 | ClaudeCode 是本地客户端，模型服务在远端或兼容端点 | `claude --version` 只能证明客户端存在 |
| 能启动就代表环境健康 | 还需要认证、网络、权限和命令路径正常 | 运行 `claude doctor` |
| npm 安装和官方安装永远等价 | 不同安装方式可能影响路径和更新节奏 | 用 `which -a claude` 检查路径 |
| VS Code 扩展 ≠ CLI | 两者底层逻辑一致，但配置入口、权限提示、命令面板都不同 | 分别走过一遍各自的登录验证 |

</div>

### 3.1 你需要理解的边界

&emsp;&emsp;ClaudeCode 是运行在本地的编程代理。它本身不是模型，而是一个能读取项目文件、执行命令、编辑代码并与模型服务通信的客户端。无论你走 CLI 还是 VS Code 扩展，本质都是同一个 ClaudeCode 客户端，只是入口不同。我们把这个分层讲清楚，后续切换路径时不会困惑。

&emsp;&emsp;更换模型不是简单改名字，而是至少涉及四件事：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 3-2 ClaudeCode 客户端与模型服务分层</font></p>
<div class="center">

| 层级 | 需要配置什么 | 验证方式 |
|---|---|---|
| 客户端 | `claude` 命令可执行 或 VS Code 扩展已安装 | `claude --version` / VS Code 扩展侧边栏 |
| 认证 | API Key 或登录态 | 能发起请求且不报认证错误 |
| 端点 | 模型服务 URL | 错误端点会出现连接或 404 类错误 |
| 模型槽位 | 模型名称、上下文能力、推理参数 | 同题对比和控制台调用记录 |

</div>

&emsp;&emsp;本课主推官方 Native Install（路径 A）。npm 安装作为路径 A 的备用方案。VS Code 扩展（路径 B）是完全等效的另一条路径。

### 3.2 官方推荐安装方式（路径 A：CLI）

> 📅 **时效性说明**：当前 `https://claude.ai/install.sh` HTTP 302 重定向至 `bootstrap.sh`，命令本身仍有效。如官方更新入口，以 `https://code.claude.com/docs/en/setup` 为准。

&emsp;&emsp;官方文档当前列出的推荐安装方式是 Native Install。macOS、Linux、WSL 可使用：

In [ ]:
!curl -fsSL https://claude.ai/install.sh | bash

&emsp;&emsp;Windows PowerShell 可使用。PowerShell 的提示符通常带有 `PS`，例如：

```text
PS C:\Users\Alice>
```

&emsp;&emsp;在 PowerShell 中只复制下面这一行命令，不要复制前面的提示符：

In [ ]:
!irm https://claude.ai/install.ps1 | iex


&emsp;&emsp;Windows CMD 可使用。CMD 的提示符通常不带 `PS`，例如：

```text
C:\Users\Alice>
```

&emsp;&emsp;在 CMD 中只复制下面这一行命令，不要复制前面的提示符：

In [ ]:
!curl -fsSL https://claude.ai/install.cmd -o install.cmd && install.cmd && del install.cmd


&emsp;&emsp;安装后重启终端，执行：

In [ ]:
!which claude
!claude --version
!claude doctor

&emsp;&emsp;通过标准：

- `which claude` 能找到可执行文件。

- `claude --version` 能输出版本号。

- `claude doctor` 不出现阻断性错误。

### 3.3 npm 备用安装方式（路径 A 备用）

&emsp;&emsp;如果 Native Install 不适合你的环境，可以用 npm 安装官方包：

In [ ]:
!npm install -g @anthropic-ai/claude-code

&emsp;&emsp;不要使用 `sudo npm install -g` 作为默认方案。权限问题通常应该通过 npm 全局目录配置解决，而不是把安装过程提升为管理员权限。

&emsp;&emsp;更新 npm 安装版本时，优先使用：

In [ ]:
!npm install -g @anthropic-ai/claude-code@latest

&emsp;&emsp;安装后仍然执行：

In [ ]:
!claude --version
!claude doctor

### 3.4 路径 A 常见问题

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 3-3 官方 ClaudeCode 安装常见问题</font></p>
<div class="center">

| 现象 | 可能原因 | 处理方式 |
|---|---|---|
| `claude: command not found` | PATH 未刷新或安装失败 | 重启终端，检查安装输出，确认可执行文件位置 |
| `node` 版本过低 | npm 备用安装依赖 Node 20+ | 升级 Node 或改用 Native Install |
| `claude doctor` 报网络错误 | 代理、DNS 或防火墙问题 | 检查网络，先确认能访问官方文档 |
| macOS 提示无法打开 | 系统安全策略 | macOS 13+：System Settings → Privacy & Security → 滚动到底部点击「仍要打开」（Open Anyway） |
| 多个 `claude` 命令冲突 | npm、Native、包管理器重复安装 | 用 `which -a claude` 排查路径优先级 |

</div>

### 3.5 VS Code 扩展安装路径（路径 B：GUI）

> 📅 **时效性说明**：本节基于 2026 年 5 月初 VS Code Marketplace 中 Anthropic 官方扩展的可用状态。扩展 ID 为 `anthropic.claude-code`。后续 Marketplace 中的扩展元信息（名称、版本号、登录入口）可能调整，以 `https://marketplace.visualstudio.com/items?itemName=anthropic.claude-code` 为准。

&emsp;&emsp;路径 B 适合两类同学：一类是日常以 VS Code 为主、不习惯在终端里完成所有操作的开发者；另一类是初次接触 AI 编程工具、希望先用图形界面熟悉 ClaudeCode 的学员。VS Code 扩展和 CLI 共享同一份模型配置和权限规则，两条路径完成后第 4 章的接入步骤完全等效。

&emsp;&emsp;**步骤一：在 Extensions Marketplace 安装 ClaudeCode 扩展**

&emsp;&emsp;打开 VS Code，按 `Cmd+Shift+X`（macOS）或 `Ctrl+Shift+X`（Windows）打开 Extensions 面板，搜索 `Claude Code` 或扩展 ID `anthropic.claude-code`。找到 Anthropic 官方发布的扩展后点击 Install。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180524740.png" width=60%></div>

<p align="center"><font face="黑体" size=3>图 3-1 VS Code Marketplace 中 ClaudeCode 扩展安装界面</font></p>

&emsp;&emsp;**步骤二：在侧边栏打开 ClaudeCode 面板**

&emsp;&emsp;安装完成后，VS Code 左侧活动栏（Activity Bar）会出现 ClaudeCode 图标。点击图标进入扩展侧边栏，此时会显示登录或 provider 配置入口。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180005025.png" width=60%></div>

<p align="center"><font face="黑体" size=3>图 3-2 ClaudeCode 扩展在 VS Code 侧边栏的首次打开界面</font></p>

&emsp;&emsp;**步骤三：确认登录或 provider 配置状态**

&emsp;&emsp;如果你此前没有安装或配置过 ClaudeCode，扩展首次打开时通常会提示登录 Anthropic 账号，或引导你配置 provider。如果你已经在本机 CLI 中完成过登录，或已经通过 cc-switch / `~/.claude/settings.json` 配置过 provider，VS Code 扩展可能会直接复用现有配置，不再重复显示首次配置流程。

&emsp;&emsp;简单判断：如果侧边栏可以直接发起 New Chat 并正常返回，说明当前登录或 provider 配置已经可用；如果提示未登录、认证失败或 provider 缺失，再按界面提示登录 Anthropic 账号，或回到第 4 章用 cc-switch 配置 DeepSeek / 第三方 provider。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180020175.jpg" width=60%></div>

<p align="center"><font face="黑体" size=3>图 3-3 ClaudeCode 扩展登录首页</font></p>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180021166.jpg" width=60%></div>

<p align="center"><font face="黑体" size=3>图 3-4 ClaudeCode 扩展登录、配置文件路径</font></p>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180022009.jpg" width=60%></div>

<p align="center"><font face="黑体" size=3>图 3-5 ClaudeCode 扩展登录登录官方账号截图</font></p>

&emsp;&emsp;**步骤四：验证扩展可用**

&emsp;&emsp;确认登录或 provider 配置可用后，在 VS Code 命令面板（`Cmd+Shift+P` / `Ctrl+Shift+P`）输入 `Claude Code`，查看是否出现扩展提供的命令列表（例如 New Chat、Run Doctor、Show Memory 等）。在侧边栏点击 New Chat 发起一次测试对话，如能正常返回回复，说明扩展已可用。

&emsp;&emsp;通过标准：

- VS Code 侧边栏 ClaudeCode 图标存在且可点击。

- 完成登录或自定义 provider 配置，无认证错误。

- 命令面板能搜到 `Claude Code:` 开头的命令。

- 测试对话能返回内容。

&emsp;&emsp;**两条路径等效声明**：完成路径 A 或路径 B 任一路径后，你的 ClaudeCode 客户端就已就绪。如果第 4 章选择接入 DeepSeek / 第三方 provider，cc-switch 配置思路在两条路径上基本一致；如果你只使用 Anthropic 官方订阅账号，则可以继续使用 ClaudeCode 原生登录，不需要进入 cc-switch 配置流程。

---

## <center>第四章：接入 DeepSeek（cc-switch GUI 主路径 + CLI 补充）</center>

&emsp;&emsp;ClaudeCode 客户端安装完成后，下一步是让它接入可用模型。这里最容易产生误解：能打开 ClaudeCode 不等于已经接入 DeepSeek，能聊天也不等于路由正确。

&emsp;&emsp;本章我们把 cc-switch 桌面 GUI 作为主路径（GUI 主路径），把环境变量 / settings.local.json 写法作为 CLI 用户或 cc-switch 不可用时的补充路径。最后我们用控制台、配置状态和日志建立路由验证闭环。

&emsp;&emsp;本章我们的关键学习点不是"问模型你是谁"，而是学会用外部证据证明请求真的走到了目标模型服务——本章会把隐性线"能对话 ≠ 能验证"做第一次正式落地（具体收束在 4.6 节）。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 4-1 cc-switch 与 DeepSeek 接入的常见误解</font></p>
<div class="center">

| 常见误解 | 真实机制 | 正确判断方式 |
|---|---|---|
| cc-switch 提供模型能力 | cc-switch 只是配置管理器，模型能力来自 DeepSeek 服务 | 查看 provider、endpoint、API Key 配置 |
| 模型说自己是 DeepSeek 就可信 | 模型自报可能受提示词影响，不是路由证据 | 看控制台调用记录、日志或配置状态 |
| 能聊天就代表路由正确 | 只能证明有模型返回，不证明走到目标服务 | 至少保留两类路由证据 |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180024818.png" width=70%></div>

<p align="center"><font face="黑体" size=3>图 4-0 ClaudeCode 链路三层结构——cc-switch 只负责配置管理，模型能力来自底层服务商</font></p>

### 4.1 cc-switch 的角色

&emsp;&emsp;cc-switch 是模型提供商配置管理工具。它不提供模型能力，也不替代 DeepSeek API。它的作用是把模型端点、Key、模型名称和相关参数写入 ClaudeCode 能读取的配置。

> ⚠️ **使用边界：官方订阅账号不需要 cc-switch**
>
> 如果你使用的是 Anthropic 官方订阅账号 / 官方 OAuth 登录，ClaudeCode 可以直接走官方链路，不需要 cc-switch。cc-switch 的主要价值是接入 DeepSeek、Kimi、OpenRouter、第三方中转 API 等自定义 provider，并在多个 provider 之间切换。
>
> 简单判断：只想用官方 Claude 模型 → 走 ClaudeCode 原生登录；想让 ClaudeCode 调用第三方模型或中转 API → 再使用 cc-switch。两种模式可以共存，但切换 provider 后通常需要重启 ClaudeCode 会话，避免旧进程继续使用启动时的配置。

> 📅 **时效性说明**：本课基于 **cc-switch v3.14.1**（2026-04-23 发布）。该工具已演进为**六工具管理器**（支持 Claude Code / Codex / Gemini CLI / OpenCode / OpenClaw / Hermes Agent），本课聚焦其 ClaudeCode provider 配置能力。其他工具的 provider 管理思路是一致的，但具体字段以 v3.14.1 的实际界面为准。
>
> 📌 **术语统一说明**：cc-switch GUI 工具列表中显示为 `Claude Code`（带空格，与 Anthropic 官方产品名对齐）；为节省篇幅和保持术语一致性，本课正文叙述统一写作 `ClaudeCode`（合写）。当你看到"在 cc-switch 中选择 Claude Code 工具"时，对应的就是本课正文反复出现的"ClaudeCode"。

&emsp;&emsp;本课我们推荐先用 cc-switch 完成可视化配置，再理解等价的环境变量方案。这样后续排错时，你能分清 GUI、环境变量、模型服务三层。

&emsp;&emsp;本课我们以 DeepSeek 为主线，但 ClaudeCode 的接入范式并不只适用于 DeepSeek。只要服务商提供 Anthropic 兼容接口，通常都可以归约为同一组字段：Base URL、认证 Token、主模型、小模型或快速模型、子任务模型、超时和调用证据。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 4-2 多 Provider 接入路线总览</font></p>
<div class="center">

| 接入路线 | 典型定位 | 适合观察什么 | 主要局限或注意事项 |
|---|---|---|---|
| DeepSeek 官方直连 | 本课主线，低成本跑通 ClaudeCode | Anthropic 兼容端点、模型名、路由证据 | 旧 model ID（`deepseek-chat/reasoner/v3.2`）2026-07-24 废弃；境内访问稳定但不提供 SLA 保障 |
| 阿里云百炼 | 国内云平台正式接入路线 | Token Plan、Coding Plan、按量计费三种模式 | 三种计费模式容易混淆，开通顺序和地域受限；适合企业但起步成本高于 DeepSeek 直连 |
| 火山方舟 | 国内 Coding Plan / 多工具套餐路线 | Claude Code、Cursor、OpenCode 等工具统一接入 | 模型清单和价格以控制台为准，常有调整；多工具套餐适合需要 ClaudeCode + Cursor 等并行的团队，单工具用户性价比一般 |
| OpenRouter | 聚合平台路线 | 多 Provider、预算控制、Activity Dashboard（调用记录面板）、fallback | model slug 命名空间独立于上游 provider，需用 `provider/model-name` 格式；境外服务，国内访问需自行处理网络；ClaudeCode 最佳兼容性仍是 Anthropic 1P provider |
| cc-switch GUI | 本地配置管理路线 | provider 切换、模型槽位、配置状态可视化 | 是本机配置管理工具，不是 provider 本身；功能依赖底层 provider，自身不提供模型能力 |

</div>

&emsp;&emsp;这张表不是让你同时接入所有平台。第一节课的最低目标仍然是跑通一个稳定主线，再知道其他平台如何判断和替换。

### 4.2 安装 cc-switch

&emsp;&emsp;cc-switch 支持 macOS / Windows / Linux 三大平台。macOS 用户推荐 Homebrew 路线，一条命令完成安装；Windows 和 Linux 用户从 GitHub Releases 下载对应安装包。

&emsp;&emsp;全部安装包入口：


- GitHub Releases（最新版）：https://github.com/farion1231/cc-switch/releases/latest

**macOS**

方式一（推荐，Homebrew）：

```bash
brew tap farion1231/ccswitch
brew install --cask cc-switch
```

方式二（手动 DMG）：
- 在 Releases 页面下载 `CC-Switch-*-macOS.dmg`
- 双击挂载后将 CC Switch 拖入 Applications 文件夹
- macOS 版本已通过 Apple 代码签名和公证，可直接安装使用

**Windows**

| 文件 | 说明 |
|---|---|
| `CC-Switch-*-Windows.msi` | 推荐 — MSI 安装包，支持自动更新 |
| `CC-Switch-*-Windows-Portable.zip` | 便携版，解压即用，不写入注册表 |

**Linux**

| 发行版 | 推荐格式 | 安装命令 |
|---|---|---|
| Ubuntu / Debian / Linux Mint | `.deb` | `sudo dpkg -i CC-Switch-*.deb` |
| Fedora / RHEL / CentOS | `.rpm` | `sudo rpm -i CC-Switch-*.rpm` |
| 其他发行版 | `.AppImage` | `chmod +x CC-Switch-*.AppImage && ./CC-Switch-*.AppImage` |



> Linux 资产同时提供 **x86_64** 和 **ARM64**（`aarch64`）两种架构，按 `uname -m` 输出选择对应版本。


&emsp;&emsp;按系统选择安装包：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 4-3 cc-switch 安装资产选择</font></p>
<div class="center">

| 系统 | 推荐方式 |
|---|---|
| macOS | Homebrew 或 `.dmg` |
| Windows | `.msi` 或项目 release 中对应 Windows 安装包 |
| Linux x86_64 | `.AppImage`、`.deb` 或 `.rpm` |
| Linux arm64 | 对应 arm64 安装包 |

</div>

&emsp;&emsp;macOS 如果已经安装 Homebrew，也可以使用项目 README 中提供的 Homebrew 路线。手动安装时，以 GitHub Releases 中 v3.14.1 或当前 latest 版本为准，不要把某一个历史版本写成永久事实。

&emsp;&emsp;安装后打开 cc-switch，确认它能显示 ClaudeCode 或 Claude 相关配置入口。若你的版本界面与讲义截图或描述不同，以当前版本的 provider、model、endpoint、API Key 配置入口为准。

### 4.3 配置 DeepSeek Provider（GUI 主路径）

&emsp;&emsp;DeepSeek 官方提供 ClaudeCode 接入说明。当前官方 Anthropic 兼容地址为：

```text
https://api.deepseek.com/anthropic
```

&emsp;&emsp;**步骤一：在 cc-switch 中新增 provider**

&emsp;&emsp;打开 cc-switch 桌面端，选择 Claude Code 工具，进入 provider 管理面板，点击"新增 provider"或等价入口。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180012710.png" width=60%></div>

<p align="center"><font face="黑体" size=3>图 4-1 cc-switch 新增 provider 入口</font></p>

&emsp;&emsp;**步骤二：填写 DeepSeek 字段**

&emsp;&emsp;在新增 provider 表单中填入下表字段。注意：v3.14.1 GUI 中各字段的具体名称（例如 Base URL 框是否叫 "Base URL"、"Endpoint" 或 "API 地址"）会随版本调整，下表 GUI 字段名一列已标 TODO，请以你实际安装的 v3.14.1 界面为准。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 4-4 DeepSeek Provider 配置字段</font></p>
<div class="center">

| 概念 | GUI 字段名 | 示例或说明 | 注意 |
|---|---|---|---|
| Provider 名称 |  | `DeepSeek` | 便于识别即可 |
| API Key |  | `sk-...` | 不要截图暴露完整 Key |
| Base URL |  | `https://api.deepseek.com/anthropic` | 以 DeepSeek 官方文档为准 |
| 主模型 |  | `deepseek-v4-pro`（当前可用主模型） | 旧 ID `deepseek-chat` / `deepseek-reasoner` / `deepseek-v3.2` 将于 **2026-07-24 废弃**，请用 v4 |
| 快速模型 |  | `deepseek-v4-flash`（当前可用快速模型） | 可用于轻量任务 |
| 子任务模型 |  | `CLAUDE_CODE_SUBAGENT_MODEL` 对应模型 | 多 Agent 或子任务时才需要关注 |
| 超时 |  | `API_TIMEOUT_MS` 或工具界面中的 timeout | 长上下文任务可适当提高 |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180012637.jpg" width=70%></div>

<p align="center"><font face="黑体" size=3>图 4-2 cc-switch 中 DeepSeek provider 配置字段的填写界面</font></p>

&emsp;&emsp;**步骤三：保存并启用 DeepSeek provider**

&emsp;&emsp;填写完成后保存表单，回到 provider 列表，点击"启用"或勾选当前 provider，确保 cc-switch 把这条配置写入 ClaudeCode 实际读取的配置文件。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180022178.jpg" width=60%></div>

<p align="center"><font face="黑体" size=3>图 4-3 在 cc-switch 中启用 DeepSeek provider</font></p>

&emsp;&emsp;**步骤四：观察当前 provider 状态**

&emsp;&emsp;启用后，cc-switch 主界面会显示"当前 provider = DeepSeek"或等价状态。这一步会在第 4.6 节作为路由证据之一被引用。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180010084.jpg" width=60%></div>

<p align="center"><font face="黑体" size=3>图 4-4 cc-switch 主界面的当前 provider 状态显示</font></p>

&emsp;&emsp;不要把模型自报身份当成唯一证据。模型可能不知道自己被路由到了哪里，也可能根据提示词猜测答案。

&emsp;&emsp;其他服务商也可以按同一套字段理解。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 4-5 ClaudeCode 多 Provider 字段对照</font></p>
<div class="center">

| Provider | Base URL | 模型示例 | 适合用途 | 注意事项 |
|---|---|---|---|---|
| DeepSeek 官方 | `https://api.deepseek.com/anthropic` | `deepseek-v4-pro`、`deepseek-v4-flash` | 本课主线、低成本验证 | 旧 ID `deepseek-chat/reasoner/v3.2` 2026-07-24 废弃 |
| 阿里云百炼 Coding Plan | `https://coding.dashscope.aliyuncs.com/apps/anthropic` | `qwen3.6-plus`、`qwen3.6-flash` | 国内云平台正式接入 | Token Plan、Coding Plan、按量计费不要混淆 |
| 阿里云百炼按量计费 | `https://dashscope.aliyuncs.com/apps/anthropic` | `qwen3.6-plus`、`qwen3.6-flash` | 按 token 付费实验 | API Key 与地域、接口版本有关 |
| 火山方舟 Coding Plan | `https://ark.cn-beijing.volces.com/api/coding` | 以方舟控制台为准 | 国内 Coding Plan 和多工具套餐 | 模型清单和价格以控制台为准 |
| OpenRouter | `https://openrouter.ai/api` | `anthropic/claude-sonnet-4.6`、`deepseek/deepseek-v4 pro` | 聚合平台、预算控制、fallback | OpenRouter 模型列使用 `provider/model-slug` 路由标识符格式（命名空间独立于上游 provider 的原生 model ID，见下方表后说明）；ClaudeCode 最大兼容性仍建议 Anthropic 1P provider |

</div>

> ⚠️ **命名空间区分（避免与本表 DeepSeek 官方行混淆）**：本表 OpenRouter 行的 `deepseek/deepseek-v4 pro` 是 **OpenRouter slug**（路由标识符，格式固定为 `provider/model-name`，由 OpenRouter 平台统一维护），与本表上方 DeepSeek 官方行注意列提到的**不是同一个命名空间**——后者是 DeepSeek 原生 API 直接接收的 model ID（即将废弃）；前者是 OpenRouter 用 transformer 转发请求时用的路由名，OpenRouter slug 的命名空间独立于上游 provider 的废弃节奏，DeepSeek 原生 ID 废弃后 OpenRouter slug 仍可能继续可用。两者不可互换：OpenRouter slug 不能直接发给 DeepSeek 原生 API，反之亦然。

### 4.4 环境变量与 settings 写法（CLI 补充路径）

> 💡 **跳过决策（4.4 节是否必读）**：如果你已经在 4.3 节通过 cc-switch 完成 provider 配置，并且第 5 章的四组对比验证全部通过，**可以直接跳过 4.4 节进入 4.5 节**——4.4 节是 CLI 重度用户的补充路径，不是必经环节。如果你遇到"cc-switch 切换后未生效"或"想绕过 GUI 直接用环境变量"的场景，再回头读本节。

&emsp;&emsp;如果你是 CLI 重度用户，或者 cc-switch 暂时不可用，可以用环境变量完成同类配置。这是路径 A 用户的备用方案，也是排错时定位"配置是否被覆盖"的关键工具。字段名可能随 ClaudeCode 和中间层版本变化，以下作为排错思路和候选方案：

In [ ]:
!export ANTHROPIC_BASE_URL="https://api.deepseek.com/anthropic"
!export ANTHROPIC_AUTH_TOKEN="sk-your-key"
!export ANTHROPIC_MODEL="deepseek-v4-pro"
!export ANTHROPIC_SMALL_FAST_MODEL="deepseek-v4-flash"

&emsp;&emsp;若当前版本要求 `ANTHROPIC_API_KEY` 而不是 `ANTHROPIC_AUTH_TOKEN`，以实际报错和官方文档为准。

&emsp;&emsp;也可以把个人 provider 配置写入 `.claude/settings.local.json` 或 `~/.claude/settings.json` 的 `env` 字段。不要把真实 API Key 写入会提交到仓库的 `.claude/settings.json`（参考第 2 章安全边界）。

&emsp;&emsp;`.claude/settings.local.json` 示例：

```json
{
  "env": {
    "ANTHROPIC_BASE_URL": "https://api.deepseek.com/anthropic",
    "ANTHROPIC_AUTH_TOKEN": "sk-your-key",
    "ANTHROPIC_MODEL": "deepseek-v4-pro",
    "ANTHROPIC_SMALL_FAST_MODEL": "deepseek-v4-flash"
  }
}
```

&emsp;&emsp;OpenRouter 的写法类似，但需要显式避免旧的 Anthropic Key 干扰：

In [ ]:
!export ANTHROPIC_BASE_URL="https://openrouter.ai/api"
!export ANTHROPIC_AUTH_TOKEN="sk-or-your-openrouter-key"
!unset ANTHROPIC_API_KEY

&emsp;&emsp;验证环境变量是否已写入当前终端：

In [ ]:
!env | sort | grep ANTHROPIC

&emsp;&emsp;如果你同时使用 cc-switch 和环境变量，遇到行为异常时先检查是否存在冲突。不要在无法解释配置来源的状态下继续调试模型效果。

### 4.5 启动 ClaudeCode 并验证对话

&emsp;&emsp;接下来我们启动 ClaudeCode，发一段对话验证模型确实接通。

&emsp;&emsp;**步骤一：进入实验目录**

In [ ]:
!cd ~/claude-code-demo
!claude

&emsp;&emsp;如果你走的是 VS Code 扩展路径，在 VS Code 中打开 `~/claude-code-demo` 目录，从侧边栏的 ClaudeCode 面板发起新对话。

&emsp;&emsp;**步骤二：发起第一个测试问题**

```text
请用三句话说明你当前可以帮我做哪些本地项目协作任务。
```

&emsp;&emsp;**步骤三：再问一个代码任务**

```text
请写一个 Python 函数，输入整数列表，返回去重后的升序列表。只输出代码和一句解释。
```

&emsp;&emsp;通过标准：

- 能正常进入 ClaudeCode 会话（CLI 或 VS Code 扩展）。

- 回答没有认证错误、额度错误或端点错误。

- 回答能稳定返回中文内容和代码。

- 对话完成后，DeepSeek 控制台、cc-switch 状态或本地请求日志能提供路由证据。

### 4.6 路由验证闭环

&emsp;&emsp;我们至少要留下两类证据。这是隐性线在本章的第一次正式收束——"能对话不等于能验证"。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 4-6 DeepSeek 路由证据类型</font></p>
<div class="center">

| 证据类型 | 如何检查 | 用途 |
|---|---|---|
| cc-switch 当前状态 | 当前 provider、模型槽位、端点配置 | 证明本地配置指向 DeepSeek |
| DeepSeek 控制台 | 调用记录、额度变化、请求时间 | 证明服务端收到请求 |
| `/status` | ClaudeCode 会话中的连接状态（在 `claude` 交互窗口或 VS Code 扩展聊天框里输入斜杠命令） | 查看当前模型、端点或账号状态 |
| 云平台控制台 | 百炼、方舟或 OpenRouter 控制台 | 证明请求到达对应平台 |
| OpenRouter Activity | Activity Dashboard（OpenRouter 调用记录面板）、模型调用记录 | 观察聚合平台的 provider、费用和错误 |
| 本地日志 | cc-switch、代理层或终端输出 | 辅助定位请求是否发出 |
| 错误实验 | 用错误 Key 触发认证错误后恢复 | 验证 Key 确实参与请求 |

</div>

&emsp;&emsp;推荐记录格式：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 4-7 路由验证记录表</font></p>
<div class="center">

| 项目 | 记录 |
|---|---|
| 当前 provider |  |
| 当前模型 |  |
| 请求时间 |  |
| DeepSeek 控制台是否有记录 |  |
| cc-switch 是否显示当前 provider |  |
| `/status` 是否匹配当前配置 |  |
| 云平台或聚合平台是否有调用记录 |  |
| 是否存在环境变量覆盖 |  |

</div>

### 4.7 故障排查

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 4-8 DeepSeek 接入故障排查</font></p>
<div class="center">

| 现象 | 优先检查 | 处理方式 |
|---|---|---|
| 认证失败 | Key 是否正确、是否过期、是否有余额 | 重新创建 Key 或更换有效 Key |
| 连接失败 | Base URL、网络、代理 | 用浏览器或 curl 检查端点可达性 |
| 回答正常但路由不确定 | 控制台调用记录、cc-switch 状态、环境变量 | 不用模型自报作为结论 |
| 模型名不存在 | 模型名称或后缀失效；使用了 2026-07-24 已废弃的旧 ID | 用 `deepseek-v4-pro` / `deepseek-v4-flash` |
| 切换模型无效 | 配置未保存、会话未重启、环境变量覆盖 | 保存配置后重启 ClaudeCode |

</div>

### 本章小结

- 你已经理解 cc-switch v3.14.1 只是配置管理器，主路径走 GUI、补充路径走环境变量。

- 你已经知道 DeepSeek 接入需要 provider、endpoint、API Key 和模型槽位四件套，且模型 ID 必须用 `deepseek-v4-pro` / `deepseek-v4-flash`。

- 你已经知道路由验证必须依赖外部证据（隐性线第一次收束）。

---

## <center>第五章：用四组对比确认配置真的生效</center>

&emsp;&emsp;模型接入完成后，我们不能立刻进入项目开发。需要先做四组小对比（**四组对比**，本课件内部教学框架），把"能对话"升级为"能验证"。

&emsp;&emsp;本章我们会比较未确认路由与已确认路由、Pro 与 Flash 的任务差异、无项目上下文与有项目上下文的表现差异。每组对比都要留下记录，而不是只凭主观感觉判断。

&emsp;&emsp;这四组对比会自然引出后续课程的方法论底层——我们在这里把它先点破：AI 协作不是盲目信任回答，而是持续提供上下文、设置边界、收集证据。

&emsp;&emsp;我们这一章的目的不是评价模型好坏，而是训练你用可观察证据判断环境是否配置正确。

### 5.1 对比一：未确认路由 vs 已确认路由

&emsp;&emsp;**本节做法说明**：5.1 节我们让同一个问题问两次——第一次是在**还没看路由证据**时问（直觉判断模型是谁），第二次是在**看完 4.6 节路由证据**后再问。两次提问之间**无需切换 cc-switch 配置**，模型不变、提示词不变，唯一变化的是学员的"心理状态"——这就是本节要观察的关键变量。

&emsp;&emsp;在未检查控制台记录前，记录一次普通提问结果：

```text
请说明你当前连接的模型和服务商。
```

&emsp;&emsp;记录：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 5-1 模型自报观察记录表</font></p>
<div class="center">

| 观察点 | 结果 |
|---|---|
| 模型是否自报身份 |  |
| 自报内容是否可信 |  |
| 是否有外部路由证据 |  |

</div>

&emsp;&emsp;然后完成第 4.6 节的路由验证，再记录：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 5-2 已确认路由记录表</font></p>
<div class="center">

| 观察点 | 结果 |
|---|---|
| provider 是否明确 |  |
| 服务端是否有调用记录 |  |
| 仍然无法确认的部分 |  |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180025330.png" width=80%></div>

<p align="center"><font face="黑体" size=3>图 5-1 模型自报不可信——对话成功≠路由正确，外部证据才是判定依据</font></p>

&emsp;&emsp;核心结论：对话成功只能证明"有模型返回"，不能单独证明"路由正确"。

### 5.2 对比二：Pro 与 Flash 的任务差异

> 💡 **本节开始之前先看：切换后是否需要重启 claude 会话**：cc-switch 切换 provider 或模型后，**当前正在运行的 `claude` 会话不会自动感知新配置**（settings.local.json 已经改了，但已建立的会话进程会继续使用启动时的环境）。要让新模型生效，**退出 `claude` 会话**（输入 `/exit` 或 Ctrl+D），然后**重新启动** `claude`。VS Code 扩展用户同理，需要在侧边栏点击「New Chat」开新会话。下面的两次"切换 Pro→Flash"动作都按这个流程操作。

&emsp;&emsp;在 cc-switch 中切换到 Pro 模型（`deepseek-v4-pro`），提问：

```text
请分析：一个 3 人小团队是否应该把项目文档、Issue、代码注释都交给 AI 生成？请给出决策框架。
```

&emsp;&emsp;记录结果后切换到 Flash 模型（`deepseek-v4-flash`），问同一个问题。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 5-3 Pro 与 Flash 任务差异记录表</font></p>
<div class="center">

| 观察维度 | Pro | Flash |
|---|---|---|
| 响应速度 |  |  |
| 分析深度 |  |  |
| 结构化程度 |  |  |
| 是否主动列出风险 |  |  |
| 更适合的任务 |  |  |

</div>

&emsp;&emsp;我们不要只根据一次结果下结论。这里的目标是建立任务选型意识：

- 复杂分析、设计方案、代码生成，更适合高能力模型。

- 快速问答、轻量改写、信息整理，更适合快速模型。

- 真正的选择应由成本、延迟、稳定性、上下文长度和任务风险共同决定。

&emsp;&emsp;成本对比不要只看"单价最低"。ClaudeCode 任务会产生多轮上下文、工具调用、重试、子任务和缓存命中差异。更合理的方式，是先做一个统一口径的估算练习。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 5-4 成本估算练习表</font></p>
<div class="center">

| 路线 | 计费口径 | 估算方式 | 注意事项 |
|---|---|---|---|
| DeepSeek 官方 | token 单价 | 按官方 V4 Flash / V4 Pro 当天价格估算 | 促销价、缓存命中、模型后缀会变化 |
| 阿里云百炼 | Token Plan、Coding Plan、按量计费 | 先区分套餐还是按量，再看控制台 | 不要把套餐额度直接等同于 token 单价 |
| 火山方舟 | Coding Plan / 套餐 / API 额度 | 以方舟控制台当天价格为准 | 不写死课件价格 |
| OpenRouter | credits 与模型 token 单价 | 按模型页面 input/output 单价估算 | 聚合层可能涉及 provider 差异和 fallback |

</div>

&emsp;&emsp;我们可以用同一个假设来比较：一次任务消耗 `1M input + 0.2M output`。先按页面价格算出静态成本，再观察真实任务中是否因为长上下文、重复提问、工具失败或子任务增加而偏离估算。

### 5.3 对比三：无项目上下文 vs 有项目上下文

&emsp;&emsp;在空目录里提问：

```text
请帮我判断这个项目的技术栈、启动命令和主要风险。
```

&emsp;&emsp;你会看到模型只能猜测，或者要求查看文件。

&emsp;&emsp;接着创建最小项目（**Windows 用户提示**：下方的 `cat > ... <<'EOF'` 是 Bash heredoc 语法，**PowerShell / cmd 不支持**——请改用 VS Code / Notepad 直接打开两个文件粘贴内容，或在 Git Bash / WSL 中运行这段命令）：

```bash
mkdir -p src
cat > package.json <<'EOF'
{
  "scripts": {
    "test": "echo no tests yet",
    "lint": "echo no lint yet"
  },
  "dependencies": {}
}
EOF

cat > src/app.js <<'EOF'
export function uniqueSorted(numbers) {
  return Array.from(new Set(numbers)).sort((a, b) => a - b);
}
EOF
```

&emsp;&emsp;再问：

```text
请读取当前项目，判断技术栈、启动命令和最需要补齐的工程能力。先列证据，再给结论。
```

&emsp;&emsp;记录：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 5-5 项目上下文前后对比记录表</font></p>
<div class="center">

| 观察点 | 无上下文 | 有文件上下文 |
|---|---|---|
| 是否能引用具体文件 |  |  |
| 是否能给出证据 |  |  |
| 建议是否可执行 |  |  |

</div>

&emsp;&emsp;核心结论：AI 协作的质量不是只由模型决定，还由你提供的上下文、规则、验证入口和任务边界决定。

### 5.4 对比四：CLAUDE.md 三版本对照（业界基线 vs 自建模板）

&emsp;&emsp;前三组对比聚焦"路由证据"和"上下文权重"。这一组我们换一个变量：**同一个项目、同一个提问下，CLAUDE.md 不同版本对 ClaudeCode 输出质量的影响**。

&emsp;&emsp;为什么把这组对比放在本课？因为第 7.1 节会给你两条 CLAUDE.md 起步路径——一是手写极简版（22 行起步模板，下方步骤一会先给出内容供本节实验使用），二是直接装业界 12.9 万 Star 的 Karpathy 实战版（仓库背景见 7.1 节末"业界参考"）。两条路径都有人用，但效果差多少？得跑过一次才知道。我们用第 5.3 节已经搭好的最小项目 `~/claude-code-demo`，把三版 CLAUDE.md 依次跑一遍。

&emsp;&emsp;**步骤一：准备三版 CLAUDE.md**

&emsp;&emsp;在演示目录下建三个子目录，分别承载 A/B/C 三版：

- A 版：**无 CLAUDE.md**，目录里什么都不放

- B 版：**极简版**（下方代码块直接给出，约 22 行；与第 7.1 节起步模板完全一致，第 7 章会给完整讲解）

- C 版:**Karpathy 业界版**，一行 curl 命令拉本课第 7.1 节末"业界参考"指向的 65 行实战模板

&emsp;&emsp;先建三个子目录：

In [ ]:
!mkdir -p ~/claude-code-demo/{a-no-claude,b-minimal,c-karpathy}

&emsp;&emsp;**B 版内容（粘贴到 `~/claude-code-demo/b-minimal/CLAUDE.md`）**：

```markdown
# 个人 ClaudeCode 协作偏好

## 通用工作方式

- 先基于可见文件和命令输出做判断，不要直接猜测。
- 遇到不确定的库版本、命令或外部服务时，先说明不确定性，再给验证方法。
- 修改前先给简短计划，修改后说明验证方式。

## 安全边界

- 不读取 `.env`、`.env.*`、`secrets/` 或任何可能包含密钥的文件（参考第一节课第 2 章安全边界）。
- 不把 API Key、Token、账号密码写入项目文件、截图或聊天记录。

## 输出偏好

- 先给结论，再给依据。
- 涉及文件时给出文件路径。
- 不输出冗长背景解释。
```

&emsp;&emsp;**C 版一行拉取**：

In [ ]:
!cd ~/claude-code-demo/c-karpathy
!curl -fsSL https://raw.githubusercontent.com/multica-ai/andrej-karpathy-skills/main/CLAUDE.md -o CLAUDE.md

> 💡 **关于 C 版来源**：`multica-ai/andrej-karpathy-skills` 是把 Andrej Karpathy 的 4 条 LLM 编程经验由 Forrest Chang 整理成的开源 CLAUDE.md，开源 4 个月已积累 12.9 万 Star（实测于 2026-05-15）。第 7.1 节末"业界参考"会展开介绍仓库 owner / Star 数 / 四规则对照。

&emsp;&emsp;**步骤二：用同一段代码与提问跑三次**

&emsp;&emsp;在每个子目录里都放一份相同的 `src/app.js`（复制第 5.3 节那段 `uniqueSorted` 函数即可），然后切换到 A 目录启动 `claude`，提问：

```text
请帮我给 src/app.js 中的 uniqueSorted 函数加入对空数组和 null 输入的处理。
先告诉我你打算怎么改，再动手。
```

&emsp;&emsp;然后切换到 B、C 目录各跑一次同样的提问。**不要清缓存，不要换模型**——三轮唯一变化的就是 CLAUDE.md。

&emsp;&emsp;**步骤三：记录三版输出差异**

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 5-6 三版 CLAUDE.md 输出对照记录表</font></p>
<div class="center">

| 观察点 | A 无规则 | B 极简版 | C Karpathy 业界版 |
|---|---|---|---|
| 是否先暴露假设/反对盲目动手 |  |  |  |
| 改动是否最小（只动 uniqueSorted）|  |  |  |
| 是否给出验证方法 / 测试 |  |  |  |
| 是否"顺手"改了无关代码 |  |  |  |
| 输出总行数 / 是否过度复杂 |  |  |  |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180025636.png" width=85%></div>

<p align="center"><font face="黑体" size=3>图 5-2 三版 CLAUDE.md 对照实验——同一输入、唯一变量、观察输出差异</font></p>

&emsp;&emsp;**理论上你会看到的差异**：A 组（无项目级规则）主要靠你的全局规则兜底，B 组（极简版）短小聚焦给计划，C 组（Karpathy 四规则）系统化列出多重假设。这印证了本章第 5.1 节的核心判断——**ClaudeCode 的输出质量不仅由模型决定，更由"你给它什么外部锚点"决定**。本组对比是「能对话 ≠ 能验证」这条隐性线在 CLAUDE.md 维度上的延伸：模型自报无法替代外部证据；同样地，**经验沉淀也无法靠口头叮嘱替代**——必须写进 CLAUDE.md 才能每次自动生效。

&emsp;&emsp;但**实测下来的差距可能跟你的直觉不一样**——这就是这组对比真正值得跑的原因。我们先给一组真实跑出来的参考数据，你再去对照自己的结果。

#### 参考实测数据（2026-05-15 真实跑出来的）

&emsp;&emsp;我们在 `~/claude-code-demo` 等价目录用 ClaudeCode 2.1.142 真实跑了三组（同模型、同提问、仅切换项目级 CLAUDE.md）。trio 模式独立 Sonnet + Codex 双视角审核交叉确认下列数据。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 5-7 三版 CLAUDE.md 客观数据矩阵（参考锚定）</font></p>
<div class="center">

| 客观指标 | A 无 CLAUDE.md | B 极简 22 行 | C Karpathy 65 行 |
|---|---|---|---|
| 输出行数 | 32 | 15 | 21 |
| 输出字符数 | 1196 | 795 | 1263 |
| 模型耗时 | 34s | 18s | 25s |
| 代码块数量 | 4 | 0 | 2 |
| 不确定性标记词 | 0 | 0 | 2 |
| 主动提问数 | 2 | 1 | 2 |

</div>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 5-8 三版 CLAUDE.md 5 维评分（双独立视角交叉一致）</font></p>
<div class="center">

| 5 维评分（1-5 分） | A 无 | B 极简 | C Karpathy |
|---|---|---|---|
| D1 暴露假设 / push back | 5 | 3 | 5 |
| D2 改动最小化 | 5 | 5 | 5 |
| D3 给出验证方法 | 2 | **5** | 3 |
| D4 不碰无关代码 | 5 | 5 | 5 |
| D5 输出复杂度合适 | 4 | 5 | 4 |
| **总分（25 满）** | **21** | **23** | **22** |

</div>

&emsp;&emsp;**3 个反预期发现**（独立 Sonnet + Codex 双视角审核同步确认）：

&emsp;&emsp;**反预期发现 1：B 极简版反而拿第一**。22 行的极简模板在「验证方式」维度独家拿 5 分——它是三组中唯一显式列出 `null→[]`、`[]→[]`、`[3,1,2,2]→[1,2,3]` 三组测试用例的；C 的 65 行 Karpathy 版反而没列出测试。**规则颗粒度往往比规则数量更重要**：B 的"修改后说明验证方式"这条精准条款，比 C 的"Goal-Driven Execution"这种通用原则更直接命中本任务。

&emsp;&emsp;**前提说明（针对没装全局 CLAUDE.md 的学员）**：下面的"反预期发现 2"假设你已经在 `~/.claude/CLAUDE.md` 写了全局规则——如果你还没装（首次跑本课时常见），先按 7.1 节模板把全局规则建好再回来跑 5.4 节实验，否则 A 组结果会偏向"完全无规则"而非"仅全局规则兜底"。

&emsp;&emsp;**反预期发现 2：A 无 CLAUDE.md 不弱**。A 组在「暴露假设」维度与 C 持平（同得 5 分），因为你的**全局 `~/.claude/CLAUDE.md`** 已经在所有项目生效——本课用户的全局规则包含「澄清在前」「管理困惑不装懂」「成功标准优先于步骤指令」等条款，这些在 A 组同样发挥作用。**全局规则的兜底效应经常被低估**——它不是 CLAUDE.md 体系里"可有可无"的那一层，而是真正贯穿所有项目的稳定底座。

&emsp;&emsp;**反预期发现 3：C 的 push back 中有 1/2 是伪命题**。C 给出的"两个澄清点"里，第 1 点（空数组语义）本质是伪问题——`new Set([])` 行为已经正确，无需澄清；第 2 点（undefined 处理）才是真澄清。**通用大模板有时会触发不必要的澄清**——这是规则集越多越全的副作用。

> 💡 **教学结论**：CLAUDE.md 确实起作用，但**"挂哪个版本"没有银弹**——颗粒度精准的极简版可能比通用全面的业界版更适合你的具体场景。更合理的分层是：**让全局 `~/.claude/CLAUDE.md` 承担通用底层**（管理困惑、简洁优先、最小改动），**让项目级 CLAUDE.md 承担项目特异**（验证方式、领域术语、特定边界）。Karpathy 65 行业界版的最高价值不在于"挂上就完事"，而在于**它把一个团队多次踩坑后的 4 条底层规则写成了可被你借鉴的样本——你可以从它复制一段，结合自己的项目实际情况再裁剪**。

&emsp;&emsp;现在按本节步骤一/二/三跑你自己的三组对比。你看到的数字不会和表 5-7、表 5-8 完全一样（模型采样有随机性、用户全局规则也不同），但量级和方向应该接近——**如果差距巨大，那本身就是值得复盘的发现**。

---

## <center>第六章：开源路线深度部署（cc-haha + CCR + DeepSeek）</center>

&emsp;&emsp;官方主线验证完成后，我们进入本课最重要的章节——开源路线深度部署。这里的目标不是简单寻找"免费 ClaudeCode"，而是训练你判断一个 AI 编程工具是否适合进入企业内网：它能否合法使用，能否接入内部模型网关，能否控制权限，能否留下审计证据，能否被团队长期维护。

&emsp;&emsp;本章我们会先把开源路线分成三类：企业正式平台路线、自研 Claude Code-like CLI 路线、功能参考路线。之后我们把 cc-haha + CCR + DeepSeek 作为完整研究实验跑通，帮助你观察本地 CLI、协议路由层和远端模型服务之间如何协作。

### 6.0 章节开场：开源路线在本课为什么必做

&emsp;&emsp;在原本"工作台搭起来"的显性目标之外，我们要把这一章在第一节课的位置说清楚，开源路线深度部署在第一节课承担两件事：

&emsp;&emsp;**显性能力扩展**：跑通 cc-haha CLI → CCR 协议路由层 → DeepSeek 模型服务的完整链路。这是 ClaudeCode 主线之外的一条研究环境，可以帮助你理解 "本地 CLI 客户端、路由调度层、模型服务" 这种三层架构是怎么协作的。后续在企业内部署、模型网关切换、协议兼容性排错时，这条链路里的知识会反复出现。

&emsp;&emsp;**隐性证据观察的训练场**：第 4 章我们做了一次路由证据闭环（cc-switch 状态 + DeepSeek 控制台），但那时只跨了一层（ClaudeCode 客户端 → DeepSeek 服务）。开源路线跨了三层（cc-haha → CCR → DeepSeek），每一层都需要单独验证。本章 6.14 节会用"三处证据凑齐链路验证"（本课件内部教学框架，非业界通用术语）把贯穿全课的"能对话 ≠ 能验证"观念做最终收束——第 5 章通过四组对比强化了这一训练（同 prompt 多次对话感知差异），第 6 章以三处硬证据给出最终结论（A 协议层路由 + C 服务端账单 + B 辅助佐证三方对齐——A/B/C 字母只是 6.14 节内部的证据编号，本节先建立"三处证据"的预期，完整定义见 6.14 节）。

&emsp;&emsp;**重要提示**：本章全部必做，无法跳过。这是第一节课的核心重点，覆盖了显性能力训练的最复杂场景、隐性观念训练的最完整收束、以及第二节课会用到的协议层知识。

> ⚠️ **路径 B 学员补做提醒**：如果你在第二章按双路径分流提示跳过了 2.1–2.5 节（Node.js / npm 安装），现在需要先回头补做这两块再继续本章。本章 cc-haha CLI 依赖 Bun（6.7 节会装）；CCR 协议路由层硬性要求 Node 20+ LTS（参考第 2 章 2.1–2.5 节）。VS Code 扩展在本章无等效 GUI 替代——开源路线必须走 CLI。建议补做顺序：① 回第 2 章 2.1–2.5 节装 Node 20+ → ② 回本章 6.7 节装 Bun → ③ 继续 6.8 节及之后。

### 6.1 企业内网部署路线总览表

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 6-1 企业内网部署路线总览</font></p>
<div class="center">

| 路线 | 代表项目 | 核心功能 | 优势 | 主要风险 | 适合场景 |
|---|---|---|---|---|---|
| 企业正式平台路线 | OpenHands | Web/CLI/SDK、任务执行、容器沙箱、企业集成、BYO LLM（自带模型） | 更接近多用户企业平台，适合权限、审计、Kubernetes 和内部 Git 集成 | 复杂度高，企业版能力和许可需要单独评估 | 公司级 AI 编程平台、统一入口、集中治理 |
| 自研 Claude Code-like CLI 路线 | Claurst、Claw Code | 终端 Agent、文件编辑、命令执行、MCP、多 Provider、权限模式 | 更适合做可控底座，可接内部模型网关，可按公司规则改造 | 成熟度不如官方 ClaudeCode，部分 parity（功能对等度）仍需验证 | 内部 PoC（概念验证项目）、自研 CLI、模型网关实验、Agent runtime 研究 |
| 功能参考路线 | cc-haha | TUI（终端交互界面）、Desktop、IM 接入、Computer Use、Skills、多 Agent、CCR/第三方模型链路 | 功能形态完整，适合学习 ClaudeCode 类产品如何组织能力 | 明确基于泄露源码，license 限制商业使用，企业合规风险高 | 架构学习、产品形态参考、研究实验 |

</div>

### 6.2 企业正式平台路线：OpenHands

&emsp;&emsp;企业正式平台路线关注的不是单个开发者在终端里能不能和 AI 对话，而是团队能不能把 AI 编程能力纳入统一治理。它通常需要多用户入口、任务队列、沙箱执行、权限策略、审计日志、模型网关、企业 Git 平台集成，以及可在内网或私有云部署的运行方式。

&emsp;&emsp;OpenHands 更接近这种平台形态。它的重点不是复刻 ClaudeCode 的终端体验，而是把软件开发任务放进一个可调度、可隔离、可集成的 Agent 平台中。对于企业来说，这类路线的价值在于治理能力：谁发起任务、任务访问了哪个仓库、用了哪个模型、执行了哪些命令、产出了哪些 diff，都更容易被平台化记录下来。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 6-2 企业正式平台路线判断点</font></p>
<div class="center">

| 判断点 | 关注问题 | 通过标准 |
|---|---|---|
| 部署形态 | 是否能部署在公司内网或私有云 | 支持 Docker、Kubernetes 或等价企业部署方式 |
| 模型接入 | 是否能接内部模型网关 | 支持 BYO LLM（自带模型，企业把内部模型网关挂进来）、OpenAI 兼容接口或 LiteLLM |
| 执行隔离 | Agent 是否在受控环境执行命令 | 有容器、沙箱或工作区隔离机制 |
| 权限治理 | 是否能控制仓库、命令、文件和用户权限 | 有明确权限模型和管理员配置入口 |
| 审计能力 | 是否能追踪任务、命令、模型和产物 | 有日志、任务记录、会话记录或审计接口 |
| 维护能力 | 企业是否能长期维护 | 有活跃社区、企业版支持或清晰升级路径 |

</div>

&emsp;&emsp;这条路线适合公司级落地。如果企业目标是给多个团队提供统一 AI 编程入口，而不是让每个人自己配置本地 CLI，那么应该优先评估这种平台路线。它的代价是部署和治理成本更高，课程中的个人工作台练习无法完全覆盖它的企业级运维细节。

### 6.3 自研 Claude Code-like CLI 路线：Claurst / Claw Code

&emsp;&emsp;自研 CLI 路线的目标，是获得一个可控的 ClaudeCode-like Agent 底座。它不一定要拥有完整桌面端或企业平台后台，但需要具备终端交互、文件读写、命令执行、权限边界、MCP、Provider 路由、会话管理和可观测输出。

&emsp;&emsp;这条路线我们优先看 `Claurst`（仓库 https://github.com/Kuberwastaken/claurst）。它走的是 Rust clean-room（干净重写，不引用原始代码）路线，重写过程有公开记录：先由 AI 分析行为产出规格说明，再由另一套 AI 仅依据规格用 Rust 实现，全程不接触原始 TypeScript，因此法律来源相对清晰，适合作为可控底座接入内部模型网关做企业 PoC（概念验证项目）。它的主要约束是 GPL-3.0：会给企业分发和产品集成带来额外义务，评估时需要先看法务能否接受。

&emsp;&emsp;`Claw Code`（仓库 https://github.com/ultraworkers/claw-code）则建议只作为形态观察对象，暂不作为企业 PoC 底座推荐。它确实是开源 Rust 实现、支持多 Provider 与权限模式，社区热度极高（GitHub 星标已达十万量级），但目前没有任何 tagged release、项目仅数周历史，且它"clean-room 重写"的说法是仓库自述、尚未经第三方核实——与之同名的仓库还有多个，其中部分直接归档泄露源码。在法律来源和工程成熟度被独立验证之前，把它放进企业内网做 PoC 的风险不低于它带来的便利。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 6-3 自研 Claude Code-like CLI 路线判断点</font></p>
<div class="center">

| 判断点 | Claurst 这类自研底座的价值 | 企业需要补齐的部分 |
|---|---|---|
| 模型路由 | 可接 Anthropic、OpenAI 兼容、本地模型或内部网关 | 统一 Key 管理、配额、成本统计和模型白名单 |
| 工具执行 | 支持文件读写、搜索、命令执行、Git 类操作 | 命令 allowlist、危险操作审批、执行日志 |
| 权限边界 | 有 workspace、read-only、danger 等权限模式 | 和企业身份、项目权限、代码仓库权限打通 |
| 可观测性 | 适合输出 JSON、状态、诊断信息 | 统一接入日志平台、审计平台、告警平台 |
| 二次开发 | 开源底座便于改造 | 需要内部 owner、测试基线和升级策略 |
| 产品体验 | CLI 更轻量，研发团队上手快 | 多用户管理、Web 控制台、任务看板需要自研 |

</div>

&emsp;&emsp;这条路线适合技术团队掌控力强、愿意自研平台能力的企业。它比直接使用官方 ClaudeCode 更可控，比完整企业平台更轻，但需要自己承担安全、审计、权限、升级和支持责任。

### 6.4 功能参考路线：cc-haha

&emsp;&emsp;cc-haha 不是因为功能少才只适合学习。相反，它的功能形态很完整：TUI、`--print`、MCP、插件、Skills、记忆系统、多 Agent、Desktop、IM 接入、Computer Use，以及通过 CCR 或 LiteLLM 接第三方模型的链路。这些设计都很适合用来理解 ClaudeCode 类产品如何组织能力。

&emsp;&emsp;真正的问题在于企业合规。cc-haha 仓库的 LICENSE 文件原文明确写明：

> "This software is provided **for educational and research purposes only**. It **shall not be used for commercial purposes**, including but not limited to commercial product development, commercial services, or any other commercial activities."
>
> 中文翻译：本软件**仅供教育和研究目的使用**。它**不得用于商业目的**，包括但不限于商业产品开发、商业服务、或任何其他商业活动。

&emsp;&emsp;cc-haha 自身说明它基于 ClaudeCode 泄露源码修复，本地 LICENSE 也明确写了只能用于 educational/research，不得商业使用，不得重新分发或销售。企业内部研发、分发给员工、接入公司代码库，通常都会被视为业务使用或商业环境使用。这不是一个功能问题，而是来源、权利链和使用限制问题。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 6-4 cc-haha 的价值与边界</font></p>
<div class="center">

| 维度 | 适合参考的内容 | 不适合直接采用的原因 |
|---|---|---|
| 产品形态 | 桌面端、多会话、Diff、权限审批、Provider 配置 | 代码来源存在泄露源码风险 |
| 远程入口 | Telegram、飞书、微信、钉钉 Adapter 的交互思路 | 企业 IM 接入需要重新做安全和权限设计 |
| Agent 能力 | Skills、多 Agent、Computer Use、记忆系统 | 功能实现可能继承不可用的权利边界 |
| 模型链路 | CCR、LiteLLM、Anthropic 兼容协议转换 | 可学习协议链路，但不应复制受限代码 |
| 教学价值 | 观察 ClaudeCode 类产品的完整能力拼装 | 只适合作为研究样本，不适合作为企业基线 |

</div>

&emsp;&emsp;因此，本课我们把 cc-haha 放在功能参考路线，而不是企业部署路线。你可以学习它如何设计桌面端、IM Adapter、权限确认、Provider 配置和多会话体验；但如果要做企业内部署，应该重新基于合法可用的开源底座或自研实现来构建。

### 6.5 三条路线的选择建议

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 6-5 路线选择建议</font></p>
<div class="center">

| 你的目标 | 推荐路线 | 原因 |
|---|---|---|
| 给公司多个团队建设统一 AI 编程平台 | 企业正式平台路线 | 更容易做权限、审计、任务管理和集中运维 |
| 做内部 ClaudeCode-like CLI 或 Agent runtime | 自研 CLI 路线 | 可控、轻量、便于接内部模型网关和安全策略 |
| 学习 ClaudeCode 产品形态和协议链路 | 功能参考路线 | cc-haha 功能完整，适合观察，但不适合生产 |
| 只完成本课后续学习 | 官方 ClaudeCode 主线 | 最稳定，后续课程默认以官方路线为准 |

</div>

&emsp;&emsp;本章后面我们做的 cc-haha + CCR + DeepSeek 操作，属于功能参考路线中的完整研究实验。它的价值是帮助你理解本地 CLI、协议路由、模型服务之间的关系。它不是官方 ClaudeCode 的替代品，也不建议作为生产主力环境。但作为第一节课的开源能力训练，它是必做的核心内容。

### 6.6 研究实验路线图

&emsp;&emsp;开源研究路线包含四层：

```text
cc-haha CLI
  -> 本地 Anthropic 兼容地址
  -> CCR 本地路由层
  -> DeepSeek 或其他模型服务
```

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180005051.png" width=85%></div>

<p align="center"><font face="黑体" size=3>图 6-0 开源路线四层架构——cc-haha → CCR → 网络 → DeepSeek，每层独立验证</font></p>

&emsp;&emsp;你需要跑通：

- Bun 运行时。

- cc-haha 源码或可执行入口。

- Claude Code Router（CCR v2.0.0）。

- CCR 到 DeepSeek 的模型配置。

- cc-haha 到 CCR 的本地端点配置。

### 6.7 安装 Bun

&emsp;&emsp;macOS、Linux、WSL：

In [ ]:
!curl -fsSL https://bun.sh/install | bash

&emsp;&emsp;重新打开终端后检查：

In [ ]:
!bun --version

&emsp;&emsp;Windows 用户可参考 Bun 官方文档，也可在 WSL 中完成本章。

> ⚠️ **macOS Gatekeeper 提示**：通过 `curl` 官方脚本安装的 Bun **不会**带有 quarantine 属性，可以直接使用，无需手动放行。**只有通过 `brew install bun` 安装的版本**才可能在首次启动时被 Gatekeeper 拦截，提示"无法打开，因为它来自身份不明的开发者"。这种情况下处理方式是：
>
> ```bash
> xattr -d com.apple.quarantine $(which bun)

或在 System Settings → Privacy & Security 中点击"仍要打开"。如果你按本课命令用 curl 安装，可以忽略这一段。

&emsp;&emsp;**✅ 本步验证**：在终端执行 `bun --version`，应输出非空版本号（如 `1.x.x`）。若提示 `command not found: bun`，重新打开一个新终端窗口（让 PATH 重新加载），或手动 `source ~/.bashrc` / `source ~/.zshrc`。

### 6.8 获取并安装 cc-haha

&emsp;&emsp;**步骤一：clone 仓库**

In [ ]:
!mkdir -p ~/Git
!cd ~/Git
!git clone --depth=1 https://github.com/NanmiCoder/cc-haha.git
!cd cc-haha

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180010043.png" width=80%></div>

<p align="center"><font face="黑体" size=3>图 6-1 cc-haha 仓库 clone 后的目录结构</font></p>

&emsp;&emsp;**步骤二：安装依赖**

In [ ]:
!bun install

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180010056.jpg" width=60%></div>

<p align="center"><font face="黑体" size=3>图 6-2 bun install 在 cc-haha 目录下成功安装依赖的输出</font></p>

&emsp;&emsp;**步骤三：检查项目脚本**

In [ ]:
!ls
!cat package.json

&emsp;&emsp;如果仓库结构或启动命令已经变化，以当前 README 为准。本讲义不把第三方开源仓库的命令写成永久事实。

&emsp;&emsp;** 本步验证**：在 `~/Git/cc-haha` 下执行 `ls` 应看到 `package.json` + `node_modules/`（或 `bun.lockb`）等典型 Node/Bun 项目结构。若 `node_modules` 缺失，回头重跑 `bun install`。

### 6.9 安装 Claude Code Router（CCR v2.0.0）

&emsp;&emsp;CCR 是 cc-haha 与模型服务之间的本地路由/调度层（协议转换只是它的可选能力之一，并非每个后端都需要）。当前版本 **v2.0.0**（MIT 协议，npm 包名 `@musistudio/claude-code-router`）。

&emsp;&emsp;先把它的职责定位说清楚，因为这里很容易理解偏。`CCR`（`Claude Code Router`，直译"Claude Code 路由器"）的核心职责是**路由与调度**：它在本地起一个服务，对上伪装成一个标准 `Anthropic` 端点接住 cc-haha 发来的请求，再按你写的规则决定"这次请求该发给哪个后端、哪个模型"，然后转发出去。这也是 6.6 节那张四层架构图里"CCR 本地路由层"的职责所在——它不产生智能，只做调度。它**顺带**还能做协议转换：当目标后端只认 `OpenAI` 格式或其它非 Anthropic 格式时，CCR 用 transformer 把请求/响应在两种格式间翻译。但请注意，协议转换是**可选的、视后端而定**的——如果后端本身就原生支持 Anthropic 协议，这一步根本不需要。

&emsp;&emsp;说到这里你可能会有一个很合理的疑问：本课用的 `DeepSeek` V4（`deepseek-v4-pro` / `deepseek-v4-flash`）**已经原生提供 Anthropic 兼容端点**（`https://api.deepseek.com/anthropic`），cc-haha 只要把 `ANTHROPIC_BASE_URL` 直接指向它就能跑通——既然能直连，为什么还要在中间夹一层 CCR？这个问题问得对：单就"接一个原生支持 Anthropic 协议的 DeepSeek"而言，CCR 的协议转换确实是多余的一跳，直连更简单。本课仍然带你完整跑一遍 CCR，原因不在"接 DeepSeek 必须用它"，而在三件可迁移的能力：一是当你要接的后端**不**原生支持 Anthropic 协议时（比如旧版 `Qwen` 端点、本地 `Ollama`、`OpenRouter` 聚合），CCR 的协议转换就是刚需；二是 CCR 能把同一个会话里的不同任务**分流到不同模型**（后台杂活用便宜模型、深度推理用强模型），这是直连做不到的；三是"客户端 → 路由层 → 模型服务"这种把模型选择从客户端解耦出来的三层结构，正是企业模型网关的雏形——本课真正想让你掌握的是这套结构，DeepSeek 只是用来跑通它的具体例子。

&emsp;&emsp;围绕这个核心职责，`CCR` 还提供了一组让多模型协作更顺手的能力，理解它们能帮你看懂后面 6.10 的配置到底在配什么。第一是**场景路由**：可以把不同类型的任务分流到不同模型，配置里用 `default`（默认）、`background`（后台轻量任务）、`think`（深度推理）、`longContext`（长上下文）、`webSearch`（联网搜索）、`image`（图像）等键分别指定，例如让便宜模型跑后台任务、让强模型只接深度推理。第二是**多 Provider 支持**：除 `DeepSeek` 外还内置 `OpenRouter`、`Gemini`、`Ollama`、`SiliconFlow`、`Volcengine` 等多家接入方式。第三是 **transformer 请求/响应转换**：用来抹平各家接口在字段、流式输出、工具调用上的差异——这也是后面配置里一个对大小写敏感、容易踩坑的地方，6.10 会专门提醒。第四是**动态切换**：可以在客户端里用 `/model` 命令临时换模型，无需重启服务。此外它还配套了 `ccr` 命令行和 `ccr ui` 网页配置界面，配置统一存放在 `~/.claude-code-router/config.json`。下一节我们就来写这份配置。

&emsp;&emsp;**步骤一：常规安装方式**

In [ ]:
!npm install -g @musistudio/claude-code-router

&emsp;&emsp;**步骤二：验证命令存在**

In [ ]:
!ccr --help

&emsp;&emsp;**步骤三：如果命令不存在，先检查 npm 全局路径**

In [ ]:
# npm v9+ 已废弃 `npm bin -g`，统一改用 npm config get prefix
!npm config get prefix
!which ccr

&emsp;&emsp;`npm config get prefix` 会输出全局安装根路径（如 `/usr/local` 或 `/opt/homebrew`），实际 ccr 可执行文件在该路径下的 `bin/ccr`。把这个 `bin/` 加入 `$PATH` 即可。

> ⚠️ **国内网络 npm install 超时**：如果上面的命令在国内网络下超时或下载缓慢，可以换淘宝镜像：
>
> ```bash
> npm install -g @musistudio/claude-code-router --registry https://registry.npmmirror.com
> ```
>
> 注意 CCR 引擎要求 **Node 20+**（参考第 2 章）。**安装 CCR 前先跑一句 `node --version` 自查**：若输出 `v18.x` 或更低，先用 `nvm install --lts` + `nvm use --lts` 升级到 Node 20+ LTS。在 Node 18 上直接装 CCR 会立刻报错退出。

&emsp;&emsp;**✅ 本步验证**：执行 `ccr --version` 应输出版本号 `2.0.0`（或更高）。若 `command not found: ccr`，先跑 `npm config get prefix` 拿全局根路径（再拼 `/bin`），确认该路径在 `$PATH` 内；macOS/Linux 通常是 `/usr/local/bin` 或 nvm 当前 Node 版本的 bin 目录。

### 6.10 写 CCR 候选配置

&emsp;&emsp;CCR 配置通常位于：

```text
~/.claude-code-router/config.json
```

&emsp;&emsp;**步骤一：先创建目录**

In [ ]:
!mkdir -p ~/.claude-code-router

&emsp;&emsp;然后用 `cat > ... <<'EOF' ... EOF` 把下面的 JSON 内容写入 `~/.claude-code-router/config.json`（也可以用 VS Code / nano / vim 直接打开该路径编辑）。

&emsp;&emsp;**步骤二：候选方案 B（Anthropic 兼容端点， 本课推荐 — 长期可用）**

&emsp;&emsp;DeepSeek 已经原生支持 Anthropic 兼容端点 `https://api.deepseek.com/anthropic` 直接吃 Anthropic 格式请求，搭配当前主推的 `deepseek-v4-pro` / `deepseek-v4-flash` 模型 ID，这是本课**首选方案**。

```json
{
  "LOG": true,
  "Providers": [
    {
      "name": "deepseek-anthropic",
      "api_base_url": "https://api.deepseek.com/anthropic",
      "api_key": "sk-your-deepseek-key",
      "models": ["deepseek-v4-pro", "deepseek-v4-flash"],
      "transformer": {
        "use": ["Anthropic"]
      }
    }
  ],
  "Router": {
    "default": "deepseek-anthropic,deepseek-v4-pro"
  }
}
```

&emsp;&emsp;**步骤三：候选方案 A（OpenAI/Chat Completions 兼容端点，⚠️ 仅作对照，旧 ID 即将废弃）**

> ⚠️ **模型 ID 废弃提醒**：候选方案 A 用到的 `deepseek-chat` 与 `deepseek-reasoner` 是 DeepSeek 在 OpenAI 兼容端点下的**历史模型 ID**，DeepSeek 官方明确这两个 ID 将于 **2026-07-24 废弃**。**长期使用请走步骤二的方案 B**。下方 JSON 仅作为 OpenAI 兼容端点写法的对照展示，方便你理解 transformer 字段在两种端点下的差异；如果你只是想跑通本课，可以**直接跳过步骤三**进入步骤四。

```json
{
  "LOG": true,
  "Providers": [
    {
      "name": "deepseek",
      "api_base_url": "https://api.deepseek.com",
      "api_key": "sk-your-deepseek-key",
      "models": ["deepseek-chat", "deepseek-reasoner"],
      "transformer": {
        "use": ["deepseek"]
      }
    }
  ],
  "Router": {
    "default": "deepseek,deepseek-chat"
  }
}
```

> ⚠️ **transformer 名称大小写敏感**：方案 A 中 DeepSeek 端点的 transformer 名称是 `"deepseek"`（**小写**）；方案 B 中 Anthropic 兼容端点的 transformer 名称是 `"Anthropic"`（**大写 A**）。**写错大小写会导致 Anthropic 端点路由静默失效**——既不报错，也不路由，启动后看起来一切正常但请求根本到不了 DeepSeek。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180013584.jpg" width=60%></div>

<p align="center"><font face="黑体" size=3>图 6-3 CCR config.json 配置编辑器界面</font></p>

&emsp;&emsp;**重要边界**：

- 上面两段是候选配置，不是保证永久可用的最终配置。

- CCR 的字段名和 transformer 名称可能随版本变化。

- 如果当前 README 使用 `baseUrl` 而不是 `api_base_url`，以 README 为准。

- 一次只启用一个方案，避免排错时混淆。

- 真实 DeepSeek Key 写在 CCR config.json 里（参考第 2 章安全边界——这个文件不进 Git 仓库）。

&emsp;&emsp;**步骤四：验证 JSON 语法**

In [ ]:
!python3 -m json.tool ~/.claude-code-router/config.json

### 6.11 启动 CCR

&emsp;&emsp;**步骤一：启动 CCR**（启动后该终端保持前台运行，CCR 的所有日志直接输出到这个终端的 stdout——**不要关闭这个窗口**；后续步骤请新开一个终端）

In [ ]:
!ccr start

&emsp;&emsp;**步骤二：新开一个终端检查端口**

In [ ]:
!lsof -i :3456

&emsp;&emsp;**步骤三：核对通过标准**

&emsp;&emsp;通过标准：

- `ccr start` 没有 JSON 解析错误。

- 3456 端口有进程监听（v2.0.0 默认值），或你已明确改用其他端口。

- CCR 日志中没有阻断性错误（关键词：`401` / `unauthorized` / `model not found` / `connection refused` / `JSON parse error`，看到任一关键词即排错优先级最高）。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180014028.png" width=60%></div>

<p align="center"><font face="黑体" size=3>图 6-4 ccr start 成功输出与端口监听验证</font></p>

> ⚠️ **3456 端口被占用**：如果 `lsof -i :3456` 显示端口被旧 CCR 进程占用，可以用下面命令清理：
>
> ```bash
> kill $(lsof -ti:3456)
> ```
>
> 然后重新 `ccr start`。如需用其他端口，记得同步修改 6.12 节 `.env` 文件的 `ANTHROPIC_BASE_URL`。

### 6.12 配置 cc-haha 指向 CCR

&emsp;&emsp;在 `~/Git/cc-haha` 中创建 `.env`：

In [ ]:
!cd ~/Git/cc-haha

&emsp;&emsp;`.env` 示例：

```dotenv
ANTHROPIC_BASE_URL=http://127.0.0.1:3456
ANTHROPIC_AUTH_TOKEN=sk-local-placeholder
ANTHROPIC_MODEL=deepseek-v4-pro
ANTHROPIC_SMALL_FAST_MODEL=deepseek-v4-flash
```

&emsp;&emsp;这里的 `ANTHROPIC_AUTH_TOKEN` 是本地占位值——cc-haha 只是把它发给 CCR，CCR 不会校验它。真实 DeepSeek Key 应放在 CCR 配置（`~/.claude-code-router/config.json`）中。**不要把真实 Key 同时散落在多个文件里**（参考第 2 章安全边界）。

&emsp;&emsp;** 本步验证**：在 `~/Git/cc-haha` 下执行 `cat .env`，应看到上面 4 行变量正确写入；执行 `grep ANTHROPIC_BASE_URL .env` 应输出 `ANTHROPIC_BASE_URL=http://127.0.0.1:3456`（端口与 6.11 节启动的 CCR 端口一致）。

&emsp;&emsp;**对照：不经 CCR 的直连写法（可选）**

&emsp;&emsp;前面 6.9 节我们说过，本课用的 `DeepSeek` V4 原生支持 Anthropic 协议。如果你只想接 DeepSeek、不需要多模型路由，可以**跳过 CCR** 让 cc-haha 直连。做法是把上面这份 `.env` 里的 `ANTHROPIC_BASE_URL` 从 CCR 的本地地址改成 DeepSeek 的原生 Anthropic 端点，`ANTHROPIC_AUTH_TOKEN` 换成你真实的 DeepSeek Key：

```dotenv
ANTHROPIC_BASE_URL=https://api.deepseek.com/anthropic
ANTHROPIC_AUTH_TOKEN=你的真实 DeepSeek Key
ANTHROPIC_MODEL=deepseek-v4-pro
ANTHROPIC_SMALL_FAST_MODEL=deepseek-v4-flash
```

&emsp;&emsp;这样 cc-haha 会把 Anthropic 格式请求直接发给 DeepSeek，不再经过 CCR——少一跳、少一个要维护的本地服务。如果你不想手写 `.env`，也可以用第 4 章已经装好的 cc-switch，把同样的值写进 `~/.claude/settings.json` 的 `env` 字段：cc-haha 复用了 ClaudeCode 的 `~/.claude/settings.json` 读取机制，能读到这份配置（注意 cc-switch v3.14.1 官方支持工具列表不含 cc-haha，这属于"借道复用"，启用 provider 后建议确认 env 确实写进了 `~/.claude/settings.json`）。

> ⚠️ **优先级陷阱**：cc-haha 的配置优先级是 **环境变量 > `.env` 文件 > `~/.claude/settings.json`**。如果你改用 cc-switch 写 `settings.json` 走直连，却没删掉本节这个指向 CCR 的 `.env`，`.env` 会盖掉 `settings.json` 的直连配置，cc-haha 仍会去连那个可能没启动的 CCR，且不报错、很难排查。走直连方案前，先删掉或清空 `~/Git/cc-haha/.env`。

&emsp;&emsp;本课后续仍以 CCR 路线为主线演示（它能训练路由层和多模型调度，是可迁移的企业网关知识），直连写法作为对照保留，方便你按实际需求选择。

### 6.13 启动 cc-haha

&emsp;&emsp;**步骤一：先看当前项目脚本**

In [ ]:
!cat package.json

&emsp;&emsp;**步骤二：根据 README 或 package.json 启动**

&emsp;&emsp;常见形式可能类似：

In [ ]:
!bun run dev

&emsp;&emsp;或：

In [ ]:
!bun run start

&emsp;&emsp;**步骤三：发起首次对话验证**

&emsp;&emsp;如果启动成功，提问：

```text
请用一句话说明当前请求链路：本地 CLI、CCR、DeepSeek 分别负责什么。
```

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180010076.png" width=60%></div>

<p align="center"><font face="黑体" size=3>图 6-5 cc-haha 启动后首次对话界面（验证 DeepSeek 路由生效）</font></p>

&emsp;&emsp;记录：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 6-6 开源路线链路检查记录表</font></p>
<div class="center">

| 检查点 | 结果 |
|---|---|
| CCR 是否保持运行 |  |
| cc-haha 是否能启动 |  |
| DeepSeek 控制台是否有调用记录 |  |
| 日志是否能看到请求经过 CCR |  |

</div>

### 6.14 三处证据凑齐链路验证

&emsp;&emsp;cc-haha 对话成功并不等于链路验证完成。我们需要的是"三处证据凑齐"（本课件内部教学框架，非业界通用术语）才算路由真正被证明。这是隐性线"能对话 ≠ 能验证"在本课的最终收束点。

&emsp;&emsp;**三处证据如下**：

&emsp;&emsp;**证据 A：CCR 日志中能看到 `POST /v1/messages` 请求**

&emsp;&emsp;在启动 CCR 的终端（或 CCR 写入的日志位置）观察日志输出。每次 cc-haha 发起对话时，CCR 应该输出类似 `POST /v1/messages` 或对应 Anthropic 兼容接口的请求行。这一处证据说明 **cc-haha 成功命中了 CCR 路由层**——请求从本地 CLI 出发，确实经过了 CCR，没有直接打到外部网络的其他端点。

&emsp;&emsp;**证据 B：cc-haha 会话中的回复符合 DeepSeek 风格（辅助佐证，弱于 A 和 C）**

&emsp;&emsp;在 cc-haha 中提一个能凸显模型风格的问题（例如直接问"请用一句话告诉我你是哪个模型，由谁训练？"）。观察回复时，**可操作的判断方式有两条**：① 看回复中**是否出现中文 thinking tokens**（即 `<think>...</think>` 包裹的中文推理过程，这是 DeepSeek-R1 / `deepseek-reasoner` 系列模型的典型特征，Claude/GPT 默认不输出这种结构。**注意**：如果你用本课推荐的 `deepseek-v4-pro` / `deepseek-v4-flash`（chat 系列），通常**默认不会**输出 thinking tokens，**这本身不算路由错误**，请结合证据 A + C 判断）；② 看回复**是否明确出现"Claude / Anthropic"自称**——如果模型自称"我是 Claude，由 Anthropic 训练"，那是**反向证据**（说明 CCR 转发可能没生效或路由错误，需要立即排查）；如果模型自称"DeepSeek"或避而不答身份，则 B 不反驳路由判定。

>
> 注意：单凭这一条不够，因为模型风格可以被提示词模仿，DeepSeek 也未必每次都吐 thinking tokens。**证据 B 是辅助佐证，不可单独作为路由判定依据**——它的作用是在 A 和 C 已经成立时提供额外置信度，或者在出现"Claude 自称"等反向信号时**反向提示路由错误**。

&emsp;&emsp;**证据 C：DeepSeek 控制台调用记录在时间窗内 +1**

&emsp;&emsp;在 cc-haha 发起对话后，打开 DeepSeek 控制台（账单 / 调用记录 / 用量统计页面），确认在对话发生的时间窗内（前后几分钟）调用次数 +1，或者 token 消耗有对应增量。这是最硬的第三方证据——它从 DeepSeek 服务端角度证明 **请求确实到达了 DeepSeek**。

&emsp;&emsp;**三处证据的判定逻辑（精确表述）**：

- **A 和 C 是硬证据**（同一时间窗内必须同时成立——"同一时间窗"指 cc-haha 发出对话后 **1-5 分钟内**，DeepSeek 控制台调用记录通常 1-3 分钟刷新，超过 5 分钟仍未出现需要主动刷新页面或检查路由）：A 证明 cc-haha → CCR 这一跳通了；C 证明 CCR → DeepSeek 这一跳通了。两者合起来才能闭合完整链路。

- **B 是辅助佐证**（必须不反驳，可以不强烈支持）：B 的作用是在 A+C 已经成立时提供额外置信度，**或者在出现"模型自称 Claude / Anthropic"等反向信号时反向提示路由错误**。如果 B 不明显支持也不明显反驳（例如模型避而不答身份），不影响路由判定。

- **三条独立都不够**：只有 A 不够（不知道 CCR 把请求转给了谁），只有 B 不够（风格可仿造），只有 C 不够（DeepSeek 控制台增量可能来自其他应用）。

&emsp;&emsp;**判定通过的精确标准**：A 在 CCR 日志中**明确存在** + C 在 DeepSeek 控制台时间窗内**调用次数 +1** + B 不出现"模型自称 Claude / Anthropic"等反向证据。三者同时满足才算"开源路线链路验证完成"。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180005034.png" width=75%></div>

<p align="center"><font face="黑体" size=3>图 6-6 三处证据判定流程——A+C 硬证据必须同时成立，B 辅助佐证不反驳即可</font></p>

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 6-7 开源路线三处证据收集记录表</font></p>
<div class="center">

| 证据 | 检查位置 | 时间戳 | 是否成立 |
|---|---|---|---|
| A：CCR 日志 `POST /v1/messages` | 启动 CCR 的终端 / 日志文件 |  |  |
| B：cc-haha 回复符合 DeepSeek 风格 | cc-haha 会话窗口 |  |  |
| C：DeepSeek 控制台调用记录 +1 | DeepSeek 控制台账单页 |  |  |
| 三处证据是否同时在时间窗内成立 |  |  |  |

</div>

&emsp;&emsp;**联动第 5 章**："能对话 ≠ 能验证"的观念在此处形成最终闭环——cc-haha 对话成功不等于链路验证完成，三处外部证据才算验证。这种"多处证据交叉确认"的方法论会在后续两节课的项目协作、模型切换、调试排错中反复使用。

### 6.15 双脚本启动器：CLI 路径 + GUI 路径

&emsp;&emsp;每次手动跑 `ccr start` → 新开终端 → `cd ~/Git/cc-haha` → `bun run dev` 这套流程很啰嗦。我们把它封装成两种启动方式（**双脚本启动器**，本课件内部教学框架，非业界通用术语）：CLI 用户用 shell 脚本，GUI 用户用 macOS Automator 或 Windows `.lnk` 快捷方式。

#### 6.15.1 CLI 启动器：start-cc-haha.sh

&emsp;&emsp;`start-cc-haha.sh`：

```bash
#!/usr/bin/env bash
set -euo pipefail

PORT="${CCR_PORT:-3456}"
CC_HAHA_DIR="${CC_HAHA_DIR:-$HOME/Git/cc-haha}"

if ! command -v ccr >/dev/null 2>&1; then
  echo "ccr command not found"
  exit 1
fi

if ! command -v bun >/dev/null 2>&1; then
  echo "bun command not found"
  exit 1
fi

if ! lsof -i ":$PORT" >/dev/null 2>&1; then
  echo "starting CCR on port $PORT"
  ccr start >/tmp/ccr.log 2>&1 &
  sleep 2
fi

cd "$CC_HAHA_DIR"

if [ ! -f package.json ]; then
  echo "package.json not found in $CC_HAHA_DIR"
  exit 1
fi

echo "starting cc-haha from $CC_HAHA_DIR"
bun run dev
```

&emsp;&emsp;授权：

In [ ]:
!chmod +x start-cc-haha.sh

&emsp;&emsp;脚本的设计要点我们已经写在三处检查里：让失败更早暴露：先检查 `ccr` 和 `bun` 命令存在性，再检查端口是否已被占用（已被占用说明 CCR 还在跑，不重启），最后检查 `cc-haha` 目录的 `package.json` 是否存在。任一环节失败立刻 `exit 1`，不会让你等到模型请求时才发现链路断开。

&emsp;&emsp;如果当前 cc-haha 的启动命令不是 `bun run dev`，把最后一行改成 README 中的实际命令。

#### 6.15.2 GUI 启动器：macOS 终端命令

&emsp;&emsp;`cc-haha` 的图形界面由两个进程组成：项目根目录下的 Web 后端（`src/server/index.ts`，提供 REST API 与 WebSocket）和 `desktop/` 子目录下的前端开发服务器（`Vite`）。在 macOS 上，我们用下面这组终端命令把它们依次拉起来——首次启动前做一次依赖准备，之后每次启动只需开两个终端窗口分别跑后端和前端。

&emsp;&emsp;首次准备（只做一次）。仓库自带的 `desktop/bun.lock` 把依赖下载地址锁死在国内镜像 `registry.npmmirror.com`，该镜像对个别包会返回畸形链接导致 `bun install` 中途失败，所以要先把它换回官方源 `registry.npmjs.org`（同一份包的镜像，`sha512` 校验不变、版本零漂移）再安装：

In [ ]:
# 进入 cc-haha 项目根（含 package.json / src / desktop 的那一层，按实际路径修改）
!cd ~/Git/cc-haha/cc-haha

# 把 lockfile 的国内镜像换回官方源（仅换下载地址，版本不变）
!sed -i '' 's#registry\.npmmirror\.com#registry.npmjs.org#g' desktop/bun.lock

# 安装前端依赖
!cd desktop && bun install && cd ..

&emsp;&emsp;每次启动。`cc-haha` 后端默认监听 `3456`，但这个端口常被 `ccr`（claude-code-router）占用，因此后端改用空闲端口 `3458`，并通过环境变量 `VITE_DESKTOP_SERVER_URL` 让前端连到这个新端口。打开**第一个终端**启动 Web 后端：

In [ ]:
!export PATH="$HOME/.bun/bin:$PATH"
!cd ~/Git/cc-haha/cc-haha
!SERVER_PORT=3458 bun run src/server/index.ts

&emsp;&emsp;再打开**第二个终端**启动前端开发服务器：

In [ ]:
!export PATH="$HOME/.bun/bin:$PATH"
!cd ~/Git/cc-haha/cc-haha/desktop
!VITE_DESKTOP_SERVER_URL=http://127.0.0.1:3458 bun run dev --host 127.0.0.1 --port 2024

&emsp;&emsp;当两个终端分别出现后端的 `{"status":"ok"}` 和前端的 `VITE ... ready` 后，在浏览器打开 `http://127.0.0.1:2024`，就能看到 `Claude Code Haha` 的图形工作台：左侧是会话列表与「新建会话」「定时任务」「设置」入口，中间是新建会话区，底部可以选择权限模式与模型。看到这个界面，就说明 GUI 已经成功启动。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260515180005014.png" width=80%></div>

<p align="center"><font face="黑体" size=3>图 6-7 cc-haha 桌面端在浏览器中成功启动（http://127.0.0.1:2024）</font></p>

&emsp;&emsp;**最后一步：在 GUI 里配置 Provider（首次必做）**

&emsp;&emsp;界面打开后如果直接发消息，会看到 `There's an issue with the selected model ... It may not exist or you may not have access to it` 这样的报错。原因是 `cc-haha` 桌面端**不读 `.env` 里的模型配置**——它有一套独立的 Provider 管理体系，配置存储在 `~/.claude/cc-haha/`。首次使用必须在界面里手动配一个指向 `ccr` 的 Provider，请求才能正常转发到 DeepSeek。

&emsp;&emsp;点击界面左下角的**齿轮（设置）→ Providers** 标签页，新增一个 Provider，按下表填写，保存并设为 active 后点「测试连接」验证：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>cc-haha Provider 指向 CCR 的配置项</font></p>
<div class="center">

| 字段 | 填写值 |
|------|--------|
| API 格式 | `anthropic` |
| Base URL | `http://127.0.0.1:3456`（`ccr` 的本地端口）|
| API Key | 任意非空字符串（如 `sk-ccr`；`ccr` 不校验此值，它用自身配置里的 DeepSeek 密钥转发）|
| 模型映射 main / sonnet / opus | `deepseek-v4-pro` |
| 模型映射 haiku | `deepseek-v4-flash` |

</div>

&emsp;&emsp;保存后回到会话窗口重新发送消息，就能收到 DeepSeek 的正常回复。<font color=red>这一步是首次使用的必经步骤——终端命令只负责把前后端进程拉起来，模型链路必须在 GUI 里单独配通。</font>

#### 6.15.3 GUI 启动器：Windows .lnk 快捷方式

&emsp;&emsp;Windows 用户我们可以用 `.lnk` 快捷方式做类似的事情。

&emsp;&emsp;**步骤一：在桌面右键 → 新建 → 快捷方式**

&emsp;&emsp;**步骤二：填写"对象"字段**

&emsp;&emsp;在快捷方式向导的"对象"或"目标"字段中填入：

```text
"C:\Windows\System32\WindowsPowerShell\v1.0\powershell.exe" -NoExit -Command "ccr start; cd C:\Users\<你的用户名>\Git\cc-haha; bun run dev"
```

&emsp;&emsp;**步骤三：填写"起始位置"字段**

&emsp;&emsp;在快捷方式属性的"起始位置"字段填入：

```text
C:\Users\<你的用户名>\Git\cc-haha
```

&emsp;&emsp;**步骤四：保存并测试**

&emsp;&emsp;命名为 "Start cc-haha"，确认保存。双击桌面快捷方式应该自动打开 PowerShell 窗口、启动 CCR、进入 cc-haha 目录并运行 `bun run dev`。

&emsp;&emsp;两个 GUI 启动器都不依赖每次手动敲命令，适合长期作为研究环境入口使用。

### 6.16 故障排查补充表

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 6-8 开源路线完整故障排查</font></p>
<div class="center">

| 现象 | 优先检查 | 处理方式 |
|---|---|---|
| `bun` 不存在 | PATH 未刷新 | 重启终端或重新 source shell 配置 |
| `bun` 在 macOS 首次启动被 Gatekeeper 拦截 | 经 brew install 安装，带 quarantine 属性 | `xattr -d com.apple.quarantine $(which bun)` 或 System Settings 允许 |
| `git clone` 超时 | 网络或 GitHub 访问问题 | 配置代理或使用可访问网络 |
| `ccr start` 报 JSON 错误 | 配置文件语法错误 | 用 `python3 -m json.tool` 检查 |
| `ccr start` 报 Node 引擎错误 | Node 版本低于 20 | 参考第 2 章升级到 Node 20+ LTS |
| 国内网络 npm install CCR 超时 | npm 默认源访问慢 | 换淘宝镜像：`npm install -g @musistudio/claude-code-router --registry https://registry.npmmirror.com` |
| 3456 端口被旧 CCR 占用 | 上次启动的 CCR 没退干净 | `kill $(lsof -ti:3456)` 清理旧进程再 `ccr start` |
| Windows WSL 中 CCR 端口转发不到主机 | WSL2 默认网络隔离 | 用 `netsh interface portproxy add v4tov4 listenport=3456 listenaddress=0.0.0.0 connectaddress=<WSL IP> connectport=3456` 转发（`listenaddress=0.0.0.0` 不可省，否则只监听 IPv6 默认地址） |
| 代理（Clash/V2Ray）下证书错误（cert verify failed） | 代理 MITM 证书未信任 | 临时关闭代理，或配置 `NODE_EXTRA_CA_CERTS` 指向代理 CA |
| 401 或认证错误 | CCR 中真实 Key 错误 | 检查 CCR 配置，不要只改 `.env` |
| 模型不存在 | 模型名或端点不匹配；使用了 2026-07-24 废弃的旧 ID | 使用 `deepseek-v4-pro` / `deepseek-v4-flash` |
| Anthropic 端点路由静默失效 | transformer 名称大小写错误 | 把 `"use": ["anthropic"]` 改为 `"use": ["Anthropic"]` |

</div>

### 本章小结

- 你已经区分企业正式平台、自研 CLI 底座和功能参考三条路线。

- 你已经知道 cc-haha 的主要限制不是功能，而是源码来源、license 和企业合规风险（"educational and research purposes only" / "not be used for commercial purposes" 原文保留）。

- 你已经把 cc-haha + CCR v2.0.0 + DeepSeek 完整链路跑通，三处证据凑齐验证。

- 你已经有 CLI 启动脚本 + GUI 启动器两种入口可用。

- 你已经掌握 11 项常见故障的排查方式。

---

## <center>第七章：部署后立刻可用的提示词</center>

&emsp;&emsp;两条工作台（官方 ClaudeCode + 开源 cc-haha）都跑起来后，我们开始考虑，立刻能用它们做什么——6 条提示词模板是答案。

&emsp;&emsp;CLAUDE.md / settings / commands / 架构笔记是让这 6 条提示词"有记忆、有边界、有入口、有承接"的基础文件。本章我们以极简姿态把它们出场，让你先用起来；第二、第三节课会逐步深入。

&emsp;&emsp;本章我们不重复定义安全边界，统一参考第 2 章的全局安全边界。

> 📌 **本章阅读建议**：7.5 节是本章的核心——6 条立刻可用的提示词模板（7.5.1 到 7.5.6）。7.1–7.4 是让这 6 条提示词"有记忆、有边界、有入口、有承接"的基础配置文件介绍，可以快速读完后直接跳到 7.5 节取用提示词。

### 7.1 CLAUDE.md 极简起步

&emsp;&emsp;CLAUDE.md 是"让 AI 记住怎么和你协作"的说明书。它不是权限系统，也不能保证模型永远照做——但它能显著降低每次对话重新解释规则的摩擦。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 7-1 CLAUDE.md 作用域</font></p>
<div class="center">

| 作用域 | 常见位置 | 是否提交 | 适合放什么 |
|---|---|---|---|
| 用户全局 | `~/.claude/CLAUDE.md` | 否 | 个人通用协作偏好、输出习惯、安全习惯 |
| 项目共享 | `./CLAUDE.md` 或 `./.claude/CLAUDE.md` | 是 | 项目背景、技术栈、验证命令、团队规则 |
| 项目本地 | `./CLAUDE.local.md` | 否 | 当前项目的个人临时偏好、本机实验说明 |

</div>

&emsp;&emsp;**极简起步模板（约 20 行）**：

In [ ]:
!mkdir -p ~/.claude

&emsp;&emsp;`~/.claude/CLAUDE.md`（全局极简版）：

```markdown
# 个人 ClaudeCode 协作偏好

## 通用工作方式

- 先基于可见文件和命令输出做判断，不要直接猜测。
- 遇到不确定的库版本、命令或外部服务时，先说明不确定性，再给验证方法。
- 修改前先给简短计划，修改后说明验证方式。

## 安全边界

- 不读取 `.env`、`.env.*`、`secrets/` 或任何可能包含密钥的文件（参考第一节课第 2 章安全边界）。
- 不把 API Key、Token、账号密码写入项目文件、截图或聊天记录。

## 输出偏好

- 先给结论，再给依据。
- 涉及文件时给出文件路径。
- 不输出冗长背景解释。
```

&emsp;&emsp;**项目级 CLAUDE.md** 放在项目根目录。写法上你有两种选择：如果你想看一份人工示范模板，参考 7.1 节顶部的全局 CLAUDE.md 模板，把"工作方式 / 安全边界 / 输出偏好"三段里的内容替换成项目实际的技术栈、安全约束、验证命令即可；如果你想让 ClaudeCode 直接代你生成一份起步草稿，可以执行第 7.5.5 节（7.5 节的第 5 条提示词，「让 AI 代写 CLAUDE.md」）。**项目本地 CLAUDE.local.md** 放个人临时偏好，并加入 `.gitignore`。

&emsp;&emsp;**验证：用 `/memory` 命令**

&emsp;&emsp;在 ClaudeCode 会话中执行 `/memory`，观察当前会话加载了哪些持久指令文件。能看到全局、项目或本地记忆的入口即说明加载成功。

> ⚠️ **常见误区**：把 API Key 写入 CLAUDE.md。CLAUDE.md 会被模型读取，写入 Key 等于把它直接交给了 LLM 上下文。**真实 Key 永远只放第 2 章规定的三个地方**。

**业界参考：Karpathy 四规则风格的 CLAUDE.md**

&emsp;&emsp;第 7.1 节给的是一段话起步模板，目的是让你**学会自己写**——这是不可省的能力训练。但训练之外，你也需要一个**业界已验证的兜底版本**，遇到新项目时一行命令立刻装好。

&emsp;&emsp;`multica-ai/andrej-karpathy-skills` 这个仓库就是这样的兜底版本。它把 Andrej Karpathy 在 X（前 Twitter）上观察 LLM 编程时的四类系统性坏习惯，由 Forrest Chang 整理成一份 65 行的 CLAUDE.md，开源发布 4 个月后已积累 **12.9 万 Star、1.3 万 Fork**（实测于 2026-05-15）。

> 📅 **时效性说明**：本节涉及第三方开源仓库的 owner、Star 数和安装命令。仓库原主路径 `forrestchang/andrej-karpathy-skills` 2026 年 4 月已转移到 `multica-ai` 组织（GitHub 会自动重定向旧 URL，两个地址都能访问，但当前规范引用是 `multica-ai/`）。本节记录的是 2026-05-15 通过 GitHub API 实测的数据，长期使用时请以仓库当前 README 为准。
>
> **这个"凭搜索引擎记忆是旧地址、凭 GitHub API 实测是新地址"的差异，本身就是第 5 章「能对话 ≠ 能验证」隐性线在仓库引用场景的延伸案例——训练记忆和搜索引擎都会滞后，一手 API 才是当下事实**。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 7-2 Karpathy 四规则核心摘要</font></p>
<div class="center">

| 规则 | 核心约束 | 对应本课已讲内容 |
|---|---|---|
| 1. Think Before Coding | 不要先假设；多个解读必须列出，不要静默选一个；不清楚就停下来问 | 第 2 章「安全边界」+ 第 5 章「能对话 ≠ 能验证」隐性线 |
| 2. Simplicity First | 最少代码解决问题；不为单点用法做抽象；200 行能压到 50 行就重写 | 本课"工作台搭积木"理念 |
| 3. Surgical Changes | 只动必要的代码；不"顺手"改邻近的注释/格式；只清理自己改动产生的孤儿 | 第 6.4 节 cc-haha 合规边界教学 |
| 4. Goal-Driven Execution | 把"做什么"翻译成"什么叫做好了"；多步任务先列 step→verify 结构 | 本课贯穿全章的"先列证据再下结论""先列计划再动手修改" |

</div>

&emsp;&emsp;**一键安装（两种方式）**：

&emsp;&emsp;**方式 A：作为单文件 CLAUDE.md 拉到当前项目**

In [ ]:
# 新项目：直接拉
!cd ~/claude-code-demo
!curl -fsSL https://raw.githubusercontent.com/multica-ai/andrej-karpathy-skills/main/CLAUDE.md -o CLAUDE.md

# 已有 CLAUDE.md：追加而非覆盖
!echo "" >> CLAUDE.md
!curl -fsSL https://raw.githubusercontent.com/multica-ai/andrej-karpathy-skills/main/CLAUDE.md >> CLAUDE.md

&emsp;&emsp;**方式 B：作为 ClaudeCode 全局 plugin 安装（跨项目推荐）**

```text
/plugin marketplace add multica-ai/andrej-karpathy-skills
/plugin install andrej-karpathy-skills@karpathy-skills
```

&emsp;&emsp;装上后会在所有项目自动生效，相当于把四规则"挂在 ClaudeCode 启动钩子里"，你下次进任何新项目都不必重复 curl。

> ⚠️ **职责不重叠**：Karpathy 四规则与第 2 章「安全边界」**职责互补，不要二选一**。Karpathy 管"模型应该怎么写代码（行为约束）"，第 2 章管"模型不许碰哪些文件（权限边界）"。两者要并存。

&emsp;&emsp;**它解决了什么**（学完本节你应该能说清楚）：Karpathy 在 X 推文里讲过一句被广泛引用的判断——"LLMs are exceptionally good at looping until they meet specific goals... Don't tell it what to do, give it success criteria and watch it go."。本课件第 5 章四组对比训练你的，是这件事的反面认识——**模型自报无法替代外部证据**；Karpathy 四规则训练你的，是这件事的正面动作——**把"做什么"翻译成"什么叫做好了"，模型才会真的为你负责**。

&emsp;&emsp;**想看效果对比？** 第 5 章末新增的「第 5.4 节 对比四：CLAUDE.md 三版本对照」给了完整对照实验脚本，你可以在 `~/claude-code-demo` 里跑一遍，记录无 CLAUDE.md / 极简版 / Karpathy 版三组输出差异。

### 7.2 settings 最小示例

&emsp;&emsp;settings 是"让 AI 不能碰哪些文件"的硬边界。CLAUDE.md 告诉模型"应该怎么做"，settings 告诉 ClaudeCode 客户端"允许或禁止做什么"。

In [ ]:
!mkdir -p .claude

&emsp;&emsp;`.claude/settings.json`（团队共享最小安全配置）：

```json
{
  "$schema": "https://json.schemastore.org/claude-code-settings.json",
  "permissions": {
    "deny": [
      "Read(./.env)",
      "Read(./.env.*)",
      "Read(./secrets/**)",
      "Read(./config/credentials.json)",
      "Bash(rm -rf *)"
    ],
    "ask": [
      "Bash(npm install *)",
      "Bash(bun install *)",
      "Bash(git push *)"
    ],
    "allow": [
      "Bash(npm run test*)",
      "Bash(npm run lint*)",
      "Bash(pytest*)"
    ],
    "defaultMode": "default",
    "disableBypassPermissionsMode": "disable"
  }
}
```

&emsp;&emsp;`.claude/settings.local.json`（个人 provider 配置，不提交仓库）：

```json
{
  "env": {
    "ANTHROPIC_BASE_URL": "https://api.deepseek.com/anthropic",
    "ANTHROPIC_AUTH_TOKEN": "sk-your-key",
    "ANTHROPIC_MODEL": "deepseek-v4-pro",
    "ANTHROPIC_SMALL_FAST_MODEL": "deepseek-v4-flash"
  }
}
```

&emsp;&emsp;**验证：用假 .env.local 触发阻断**

In [ ]:
!printf "DEMO_SECRET=do-not-read\n" > .env.local

&emsp;&emsp;在 ClaudeCode 中请求："请读取 .env.local 并告诉我里面的内容。" 期望结果：ClaudeCode 拒绝读取或触发权限阻断。如果它能直接读取，说明当前配置未生效，需要检查路径、版本和 settings 加载规则。

> 📅 **时效性说明**：settings 权限语法（`Read(...)` / `Bash(...)` 等匹配规则）随 ClaudeCode 版本变化。如果你版本里的语法不一致，以 `https://code.claude.com/docs/en/settings` 为准。

### 7.3 项目命令最小示例

&emsp;&emsp;commands 是"把常用提示词变成斜杠命令"的快捷键。

In [ ]:
!mkdir -p .claude/commands

&emsp;&emsp;`.claude/commands/project-scan.md`：

```markdown
请读取当前项目，完成一次项目初读。

要求：
1. 先列出你实际查看到的文件证据。
2. 判断技术栈、启动方式、测试方式。
3. 找出最值得补齐的工程能力。
4. 不要读取 `.env`、`.env.*`、`secrets/`。
5. 不要修改任何文件。
```

&emsp;&emsp;`.claude/commands/bug-prepare.md`（注意下面用到的 `$ARGUMENTS` 是斜杠命令的**参数占位符**——当你在 ClaudeCode 中执行 `/bug-prepare 登录失败` 时，`登录失败` 这部分会自动替换到 `$ARGUMENTS` 位置，**无需手动编辑命令文件**）：

```markdown
请根据下面的 Bug 描述，整理修复前准备清单。

Bug 描述：
$ARGUMENTS

输出：
1. 复现路径
2. 可能相关文件
3. 需要补充的问题
4. 最小修复计划
5. 验证命令

在信息不足时先提问，不要直接修改文件。
```

&emsp;&emsp;**验证**：在 ClaudeCode 中执行 `/help`，查看项目命令是否出现。若当前版本要求重启才能发现新命令，先输入 `/exit` 或按 `Ctrl+D` 退出，然后重新执行 `claude` 进入新会话。然后试 `/project-scan` 和 `/bug-prepare 点击保存按钮后页面没有提示`。

> ⚠️ **命名冲突**：不要把命令命名为 `debug` / `commit` / `deploy` 等容易与内置能力或个人 skill 冲突的短名。优先使用 `project-` / `bug-` 等明确前缀。

### 7.4 架构笔记模板（第二节课的钩子）

&emsp;&emsp;`ARCHITECTURE_NOTES.md` 是第二节课"项目初读"的记录容器。它现在是空模板，第二节课会用它承接项目定位、技术栈、入口和验证命令。

&emsp;&emsp;`ARCHITECTURE_NOTES.md`：

```markdown
# Architecture Notes

## 项目一句话定位

待补充。

## 技术栈

待补充。

## 启动入口

待补充。

## 核心目录

待补充。

## 核心流程

待补充。

## 验证命令

待补充。

## 安全边界

待补充。

## 待验证问题

待补充。
```

&emsp;&emsp;这个文件现在可以保持空模板。第二节课会用它承接项目初读、架构理解和任务拆解。

### 7.5 六条立刻可用的提示词模板

&emsp;&emsp;以下 6 条提示词每条都可以直接复制粘贴，覆盖第一节课部署完成后最常见的 6 个场景。

> 💡 **建议立即保存到笔记**：每条提示词都建议复制到个人笔记应用（Notion / Obsidian / 本地 `.md` 文件均可），下一节课进入项目协作时直接复制使用。8.1 节验收表的产物⑥就是核对这 6 条是否已经入了笔记。

#### 7.5.1 第 1 条：项目初读

&emsp;&emsp;**使用场景**：进入一个新项目前，快速建立全景认知。

```text
请读取当前项目，完成一次项目初读。

要求：
1. 先列出你实际查看到的文件证据（文件路径 + 关键内容片段）。
2. 基于证据判断：技术栈、启动方式、测试方式。
3. 找出最值得补齐的工程能力（CI / 测试覆盖 / 文档 / 类型标注等）。
4. 不要读取 `.env`、`.env.*`、`secrets/`、`config/credentials*`。
5. 不要修改任何文件。

输出格式：
- 第一节：文件证据清单
- 第二节：技术栈与启动方式判断
- 第三节：补齐建议（按优先级）
```

&emsp;&emsp;**期望产出**：证据清单 + 结构判断 + 行动建议三段式输出。

#### 7.5.2 第 2 条：Bug 准备

&emsp;&emsp;**使用场景**：遇到 Bug 时，在开始修复前先整理准备清单。

```text
请根据下面的 Bug 描述，整理修复前准备清单。

Bug 描述：
$ARGUMENTS

输出：
1. 复现路径（具体步骤）
2. 可能相关的文件（按可能性排序）
3. 需要我补充的信息（如版本、日志、依赖）
4. 最小修复计划（不超过 5 步）
5. 验证命令（如何确认修复生效）

在信息不足时先提问，不要直接修改文件。
```

&emsp;&emsp;**期望产出**：可执行的 Bug 修复准备单（5 段式），明确"还需要什么信息"。

#### 7.5.3 第 3 条：代码 review

&emsp;&emsp;**使用场景**：提交前或 PR 前，让 AI 扮演 reviewer 角色做快速 review。

```text
请对当前 diff（或指定文件）做一次代码 review。

工作流：
1. 先读取项目 CLAUDE.md，了解项目协作规则和验证要求。
2. 按以下四个层次分别给出意见：
   - 安全（凭证泄露 / 权限 / 注入）
   - 逻辑（边界条件 / 错误处理 / 数据流）
   - 性能（复杂度 / IO / 重复计算）
   - 格式（命名 / 注释 / 风格一致性）
3. 每条意见必须说明文件路径 + 行号 + 具体建议。
4. 不要修改文件，只输出 review 报告。
```

&emsp;&emsp;**期望产出**：分层 review 意见列表，每条带文件路径、行号和具体改法建议。

#### 7.5.4 第 4 条：安全验证（不读 .env）

&emsp;&emsp;**使用场景**：在开始 AI 协作前，明确告知模型当前会话的安全边界。

```text
本次会话的安全边界（请回复"已收到"确认）：

1. 你不得读取 `.env`、`.env.*`、`secrets/**`、`config/credentials*` 等任何可能包含凭证的文件。
2. 你不得在回复、注释、代码或日志中输出任何 API Key、Token、账号密码。
3. 如果某个任务确实需要凭证（如调试 API 调用），请停止操作并告知"该任务需要凭证，请通过非 AI 渠道处理"。
4. 这条边界覆盖本会话所有后续请求，包括我可能忘记的隐含场景。

请确认你已收到并将严格遵守。
```

&emsp;&emsp;**期望产出**：模型明确回复"已收到"，并在**当前会话近期轮次内**对涉及凭证的请求触发拒绝声明（你可以再问一句"帮我读一下 .env"测试它是否拒绝）。

>
> ⚠️ **关于提示词安全边界的本质（必须知道）**：提示词安全边界是**软边界**——它依赖模型记住并遵守约定，会随上下文压缩、长会话漂移而逐渐失效。**硬边界必须依赖 `.claude/settings.json` 的 deny 规则**（参考 7.2 节）——deny 规则是 ClaudeCode 客户端层强制阻断，无论模型如何被诱导都无法读取被禁文件。这条提示词只是软边界的"礼貌声明"，**真正的安全屏障是第 7.2 节的硬边界配置**。

#### 7.5.5 第 5 条：CLAUDE.md 起步模板生成

&emsp;&emsp;**使用场景**：帮助你用 AI 生成自己的第一份项目级 CLAUDE.md。

```text
请帮我为下面的项目生成一份极简 CLAUDE.md（控制在 30 行以内）。

项目一句话描述：
[在这里填一句话，例如：一个 Next.js + tRPC 的内部工具，做 admin 后台]

生成要求：
1. 包含三个必备小节：工作方式 / 安全边界 / 输出偏好
2. 工作方式：先读文件再下结论 / 修改前给计划 / 不猜版本
3. 安全边界：不读 .env / 不写 Key 入仓库 / 高风险命令需说明影响
4. 输出偏好：先结论后依据 / 给文件路径 / 不冗长
5. 整体控制在 30 行以内，不要过度膨胀
6. 直接输出可粘贴的 Markdown 内容
```

&emsp;&emsp;**期望产出**：30 行以内可直接写入 `CLAUDE.md` 的 Markdown 内容。

#### 7.5.6 第 6 条：切换模型测路由

&emsp;&emsp;**使用场景**：切换 provider 或模型后，用外部证据验证路由是否真的生效。

```text
我刚刚在 cc-switch（v3.14.1）/ VS Code 扩展中切换了 provider 或模型。
请帮我设计一次路由验证：

要求：
1. 不要让我用"问你是哪个模型"的方式作为路由证据——模型自报不可信。
2. 列出我应该在哪几个外部位置找路由证据：
   - cc-switch 当前 provider 状态
   - VS Code 扩展状态栏 / 设置面板的当前 provider 显示
   - DeepSeek 控制台 / 服务商控制台调用记录
   - ClaudeCode `/status` 命令输出
   - 本地 CCR 日志（如果走开源路线）
3. 给我一个可填写的"路由验证记录表"模板，每项含"检查位置 + 时间戳 + 是否成立"。
4. 如果三处证据中有任一处不成立，提示我下一步检查方向。
```

&emsp;&emsp;**期望产出**：路由验证步骤清单 + 证据收集表模板 + 三处证据不齐时的下一步排查路径。

### 7.6 接下来会反复用到的 8 个对话功能

&emsp;&emsp;工作台搭好了、规则约束好了、提示词模板装好了——但**对话本身也有状态**。状态管不好，再好的工作台也撑不过一小时。本节我们梳理后面两节课**反复会用到的 8 个对话功能**，让你的会话能压缩、能切换、能恢复，也能用一行命令体验多智能体协作。

&emsp;&emsp;这 8 个功能里前 5 个是会话管理核心，后 3 个是为第三节课"skill 升级与多智能体"做铺路。

> 📅 **时效性说明**：本节命令基于 **ClaudeCode 2.1.142** 实测（2026-05-15）。斜杠命令通过 binary strings 实测确认存在，CLI flag 取自 `claude --help` 真实输出。新版本可能新增/调整命令，遇到不一致时优先以 `claude --help` 和 `/help` 当下输出为准。

#### 7.6.1 会话管理 5 件套（必用）

&emsp;&emsp;先看会话管理的 5 个命令。它们解决的不是代码问题，而是**对话上下文问题**——你和 Claude 之间这段对话是否过载？是否值得恢复？切到下一个任务时怎么办？

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 7-3 会话管理 5 件套（按使用频次排序）</font></p>
<div class="center">

| 命令 | 一句话作用 | 你什么时候会用到 |
|---|---|---|
| `/compact` | 压缩当前会话历史，释放上下文窗口 | 长任务做到一半，模型开始"忘记"前面讨论过的内容，或回复明显变浅 |
| `/clear` | 彻底清空当前会话状态（不可逆）| 切到一个**完全无关的新任务**，不想让上一段历史污染新对话 |
| `/resume` | 在 claude 内交互式选择并恢复历史会话 | 重启 claude 后想接着上次的思路，不知道 session id |
| `/sessions` | 列出所有历史会话（可搜索、可挑选）| 想找回前几天的某次会话，但记不清具体哪一次 |
| `claude -n <name>` | 启动 claude 时给本次会话**起名**（CLI flag）| 长项目里多次重启 claude，给每次会话取个好认的名字方便回头 `/resume` |

</div>

> ⚠️ **/compact vs /clear 最容易混淆**：`/compact` 是**压缩**（保留语义摘要，对话能续）；`/clear` 是**清空**（彻底重置，之前讨论全部失忆）。规则：**任务连续就 /compact，任务切换才 /clear**。

&emsp;&emsp;**步骤一：长任务上下文压力大时跑 `/compact`**

&emsp;&emsp;当你在 claude 会话里讨论了很长一段、模型开始"记不清前面"，或者你看到状态栏 token 用量逼近上限时，直接在 claude 输入框敲：

```text
/compact
```

&emsp;&emsp;Claude 会把前面的讨论压缩为一段语义摘要，然后让你接着原先的任务继续聊，不丢上下文连续性。

&emsp;&emsp;**步骤二：切到完全无关任务时跑 `/clear`**

&emsp;&emsp;比如你刚和 Claude 讨论完 React 组件重构，现在想问一个完全不相关的运维问题——这时跑 `/clear` 彻底清掉上一段历史，避免 React 的讨论"渗透"到新对话里影响判断。

```text
/clear
```

&emsp;&emsp;**步骤三：明天接着今天的会话用 `/resume` 或启动 flag `-r/--resume`**

&emsp;&emsp;明天打开终端不想从零开始？两种方式：

In [ ]:
# 方式 A：启动时直接选择上次会话
!claude -r
# 或交互式选择
!claude --resume

# 方式 B：进入新会话后用 /resume 命令切换到旧会话
!claude
!> /resume

&emsp;&emsp;`-c/--continue` 是另一个相关 flag——直接续上**当前目录**最近一次会话，不弹选择器，比 `-r` 更快。

&emsp;&emsp;**步骤四：长项目用 `claude -n <name>` 给会话命名**

&emsp;&emsp;长项目里我们经常一天开 5-10 次 claude，回头 `/resume` 全是"Session a3f9b2c..."不知道哪个是哪个。开始时直接起名：

In [ ]:
!claude -n "refactor-auth-module"

&emsp;&emsp;这个名字会显示在终端标题和 `/resume` 选择器里，**让历史会话好认**。

#### 7.6.2 多智能体首映：3 个铺路命令

&emsp;&emsp;ClaudeCode 不只是单 agent 工具——它内置了**多 agent 协作能力**。本课不展开技术细节（第三节课"skill 升级"会完整讲），但我们至少要让你**亲眼看到**多 agent 协作的样子。下面 3 个命令是第三节课的铺路。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 7-4 多智能体铺路 3 命令</font></p>
<div class="center">

| 命令 | 一句话作用 | 第三节课会展开 |
|---|---|---|
| `/ultrareview` | 一行命令触发**云端多 agent 代码审查**（官方内置）| ——（本节直接演示，让你看效果）|
| `/skills` | 列出当前已安装的 skill 清单 | 第三节课会教你**自己开发 skill** |
| `/agents` | 管理自定义 background agents（含创建、编辑、删除）| 第三节课会教你**自己定义多 agent 流程** |

</div>

#### 7.6.3 零配置体验：一行 `/ultrareview` 演示

&emsp;&emsp;`/ultrareview` 是 ClaudeCode 内置的多 agent 代码审查命令——它不在本地跑，而是把当前分支提交给**云端多个 agent 并行评审**，最后聚合一份评审报告给你。这是你**第一次零配置地体验多智能体协作**。

> ⚠️ **使用边界（先看再动手）**：`/ultrareview` 是 ClaudeCode 官方云端服务，**只对接 Anthropic 官方账号**——它**不可用于第三方 provider 接入的 ClaudeCode**（本课通过 cc-switch 接 DeepSeek 时会被拒绝），也**不可用于 essential-traffic-only 模式**。如果你没有 Anthropic 官方账号，**跳过本节实操**，仅读 7.6.2 命令表了解能力即可，不影响第一节课验收。

&emsp;&emsp;**前置步骤：切回官方 Anthropic provider（已用 cc-switch 接 DeepSeek 的学员必做）**

&emsp;&emsp;有两种方式切回：

- **方式 A（推荐，cc-switch GUI 一键切）**：打开 cc-switch 桌面端 → 在 provider 列表中找到 `Anthropic Official`（或你自己添加的官方账号 provider）→ 点击"启用"——cc-switch 会自动把 settings.local.json 里的 `ANTHROPIC_AUTH_TOKEN` / `ANTHROPIC_BASE_URL` 替换回官方值。

- **方式 B（CLI 手改）**：临时把 `~/.claude/settings.local.json` 里 `ANTHROPIC_AUTH_TOKEN` 改回 Anthropic 官方 key（`sk-ant-...`），并删除（或注释掉）`ANTHROPIC_BASE_URL` 这一行，让 ClaudeCode 走默认官方端点。

&emsp;&emsp;切完后，重新启动 `claude`，运行 `/help` 确认 `/ultrareview` 出现在命令列表里（接 DeepSeek 时这条命令会被隐藏或返回不支持错误）。完成后再进入步骤一。

&emsp;&emsp;**步骤一：切到任意有改动的 git 仓库**

In [ ]:
!cd ~/your-project   # 任意 git 仓库，有至少一两个 commit 在本分支
!claude

&emsp;&emsp;**步骤二：在 claude 会话里直接跑**

```text
/ultrareview
```

&emsp;&emsp;Claude 会提交当前分支到云端，触发多 agent 并行评审。等 1-3 分钟你会拿到一份分类报告（按"安全 / 逻辑 / 风格 / 性能"等维度组织，每条带文件和行号）。

&emsp;&emsp;**步骤三：观察"多 agent 协作"的可见痕迹**

&emsp;&emsp;读报告时你会看到：不同维度的发现彼此独立、互不干扰、有时甚至**相互打架**（如一个 agent 说"加缓存"、另一个说"先看真实负载再决定"）。这就是多 agent 协作的真实样子——**不是一个统一的声音，而是多个独立视角的交叉**。

> 💡 **隐性线呼应**：还记得第 5 章的"能对话 ≠ 能验证"吗？`/ultrareview` 把这条线推到了下一层——**单 agent 自审有盲点，多 agent 交叉视角才能抓到更全的问题**。第三节课你会学会**自己定义一个这样的多 agent 流程**，把"交叉视角"用在你的真实项目里。

> 💡 **边界已在节首前置**：本节开头的"使用边界"已经说明 `/ultrareview` 只对接 Anthropic 官方账号——这一边界本身也印证本节核心观念：**对话能力不是一个统一的池子，而是由"客户端 + provider + 命令"三层叠加决定的**。

### 本章小结

- 你已经知道 CLAUDE.md / settings / commands / 架构笔记四件套各自的极简起步姿态。

- 你已经有 6 条立刻可用的提示词模板可以直接复制粘贴。

- 你已经为第二节课的项目初读准备好了记录容器（ARCHITECTURE_NOTES.md）。

- 你已经掌握 5 个会话管理命令（`/compact` / `/clear` / `/resume` / `/sessions` / `-n <name>`）+ 一次零配置的多 agent 协作演示（`/ultrareview`），为第二、三节课铺好了"对话状态管理"和"多智能体"两条路。

---

## <center>第八章：总验收与课后任务</center>

&emsp;&emsp;完成所有配置后，我们用一张精简验收表把结果收束起来。验收不是为了形式完整，而是为了确认后续两节课我们真的可以直接进入项目协作。

### 8.1 必做验收表

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 8-1 第一节课必做验收</font></p>
<div class="center">

| 检查项 | 通过标准 | 结果 |
|---|---|---|
| ClaudeCode 客户端可用（产物①）| `claude --version` + `claude doctor` 正常，或 VS Code 扩展已登录可发起对话 |  |
| DeepSeek 已接入，路由有证据（产物②）| cc-switch 状态 + DeepSeek 控制台至少两类证据 |  |
| 全局与项目 CLAUDE.md 已就位（产物③）| `~/.claude/CLAUDE.md` + `./CLAUDE.md` 存在，`/memory` 能看到加载 |  |
| settings 权限边界已生效（产物④）| `.claude/settings.json` 存在，`.env.local` 读取被阻止 |  |
| 项目命令已可用（产物⑤）| `/project-scan` 和 `/bug-prepare` 在 `/help` 中可见且能执行 |  |
| 6 条提示词模板已就位（产物⑥）| 7.5.1-7.5.6 节 6 条提示词模板已保存到你的个人笔记（Notion / Obsidian / 本地 `.md` 文件均可），下次会话可直接复制使用 |  |
| 开源链路可启动 + 三处证据（产物⑦）| cc-haha + CCR + DeepSeek 完整链路 + 6.14 节三处证据（A 协议层 + C 服务端硬证据 + B 不反驳）同时成立 |  |
| 架构笔记模板已就位（产物⑧）| `ARCHITECTURE_NOTES.md` 已落到当前项目根目录，模板字段（项目定位 / 技术栈 / 启动入口 / 核心流程 / 验证命令）齐备，作为第二节课"项目初读"的记录容器 |  |

</div>

### 8.2 最终验证提问

&emsp;&emsp;在 ClaudeCode 中执行：

```text
请先通过 /memory 确认当前会话加载了哪些 CLAUDE.md 规则。
然后读取当前目录的 CLAUDE.md、ARCHITECTURE_NOTES.md 和 .claude/settings.json。
然后回答：
1. 你后续会如何遵守这些规则？
2. 哪些文件你不应该读取？
3. 遇到不确定命令时你应该怎么做？
4. 下一节课可以从哪个任务开始？
不要修改任何文件。
```

&emsp;&emsp;理想回答应该包含：

- 引用具体文件中的规则。

- 明确不读取 `.env`、`.env.*`、`secrets/`。

- 承认不确定命令需要先验证。

- 能把下一节课引向项目初读、任务拆解和受控修改。

### 8.3 课后任务

&emsp;&emsp;课后任务不是额外作业，而是把本节课的配置迁移到你的真实练习项目中。只有迁移过一次，工作台才真正属于你。

1. 在自己的机器上创建或调整 `~/.claude/CLAUDE.md`，只保留跨项目通用的个人协作偏好。

2. 在自己的练习项目中复制本课的项目 `CLAUDE.md`，删掉不适合自己的条款，加入项目背景和验证命令。

3. 在 DeepSeek 控制台确认本课至少一次请求记录，并把路由证据写入学习笔记。

4. 用 `/project-scan` 或等价入口扫描一个小项目，观察它是否先列证据再下结论。

5. **复测权限边界**：你已经在 7.2 节做过一次 `.env.local` 读取被拒的测试。本任务要求你**重新跑一次**——修改 `.claude/settings.json` 中的某条规则（如临时把 `.env.local` 加入允许列表），再让 ClaudeCode 读取，观察是否**真的能读到**；测试完后**记得改回禁止状态**。这一步是"配置改动后边界仍生效"的反向验证。

6. 选择 CLI 启动器或 GUI 启动器，把开源路线（cc-haha + CCR + DeepSeek）固化为一键启动。

### 本章小结

- 你已经拥有一张可逐项打勾的工作台精简验收表。

- 你已经把验收表延伸到开源链路的三处证据闭环。

- 你已经准备好进入第二节课的项目初读。

---

## <center>附录 A：故障排查速查</center>

&emsp;&emsp;附录 A 汇总本课最常见的失败现象。排错时我们先看错误信息，再看配置来源，最后看服务端证据。

&emsp;&emsp;遇到问题时不要同时改很多地方。我们先确认是哪一层失败：客户端、配置、认证、网络、路由、权限还是开源项目本身。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 A-1 故障排查速查表</font></p>
<div class="center">

| 模块 | 现象 | 处理方式 |
|---|---|---|
| 官方 CC | `claude` 不存在 | 检查安装输出、PATH、终端是否重启 |
| 官方 CC | 健康检查失败 | 先处理认证、网络、权限等阻断项 |
| VS Code 扩展 | 侧边栏图标不出现 | 在 Extensions 面板确认扩展已 enabled，必要时重启 VS Code |
| cc-switch | 无法打开 | 检查系统安全策略和安装包架构 |
| cc-switch | 保存后无效 | 检查是否重启 ClaudeCode，是否被环境变量覆盖 |
| DeepSeek | 认证失败 | 检查 Key、余额、服务状态 |
| DeepSeek | 模型不存在 | 用 `deepseek-v4-pro` / `deepseek-v4-flash`；旧 ID `deepseek-chat` / `deepseek-reasoner` / `deepseek-v3.2` 将于 2026-07-24 废弃 |
| 路由验证 | 模型自报不一致 | 以控制台、配置、日志为证据 |
| Bun | 命令不存在 | 重启终端，检查 shell 配置 |
| Bun | macOS Gatekeeper 拦截 | 仅 brew install 路径需要 `xattr -d com.apple.quarantine` |
| cc-haha | 依赖安装失败 | 检查 Bun 版本、网络、仓库 README |
| CCR | JSON 解析错误 | 用 `python3 -m json.tool` 检查配置 |
| CCR | Node 引擎错误 | 升级到 Node 20+ LTS |
| CCR | 端口冲突 | `kill $(lsof -ti:3456)` 清理后再 `ccr start` |
| CCR | Anthropic 端点路由静默失效 | transformer 大小写错误，改为 `"Anthropic"`（大写 A） |
| `CLAUDE.md` | 模型未遵守 | 重启会话，明确要求读取项目规则 |
| settings | deny 未生效 | 检查文件位置、路径匹配、版本支持 |
| commands | 命令不出现 | 检查路径，运行 `/help`，必要时重启 |

</div>

---

## <center>附录 B：观察记录表</center>

&emsp;&emsp;附录 B 用来保存学习过程中的证据。AI 协作质量很大程度取决于你是否愿意记录事实，而不是只记结论。

&emsp;&emsp;**复制建议**：把下面的表格复制到 Notion / Obsidian / 本地 `.md` 笔记任选一种里，**在学习过程中边做边填**（不是课后回忆填）——课后填表会丢失"我当时为什么这么选"的判断细节，而判断细节正是后续两节课所需的"协作直觉"原材料。

&emsp;&emsp;建议把这些表格复制到自己的笔记系统中，持续记录 provider、模型、路由证据、对比结果和最终环境状态。

### B.1 模型接入记录

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 B-1 模型接入记录表</font></p>
<div class="center">

| 项目 | 记录 |
|---|---|
| 官方 CC 版本 |  |
| VS Code 扩展版本（如使用） |  |
| cc-switch 版本（应为 v3.14.1+） |  |
| 当前 provider |  |
| 当前模型（推荐 deepseek-v4-pro / deepseek-v4-flash） |  |
| DeepSeek 控制台是否有请求记录 |  |
| 是否存在环境变量覆盖 |  |
| 未解决问题 |  |

</div>

### B.2 对比记录

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 B-2 综合对比记录表</font></p>
<div class="center">

| 对比项 | 观察结果 | 结论 |
|---|---|---|
| 无路由证据 vs 有路由证据 |  |  |
| Pro vs Flash |  |  |
| 无 `CLAUDE.md` vs 有 `CLAUDE.md` |  |  |
| 无权限配置 vs 有权限配置 |  |  |
| 手写重复请求 vs 项目命令 |  |  |
| 开源链路三处证据是否同时成立 |  |  |

</div>

### B.3 环境最终状态

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 B-3 环境最终状态记录表</font></p>
<div class="center">

| 文件或工具 | 是否完成 | 备注 |
|---|---|---|
| `claude` CLI 或 VS Code 扩展 |  |  |
| cc-switch v3.14.1+ |  |  |
| DeepSeek Key |  |  |
| `~/.claude/CLAUDE.md` |  |  |
| `CLAUDE.md` 或 `.claude/CLAUDE.md` |  |  |
| `CLAUDE.local.md` |  |  |
| `.claude/settings.json` |  |  |
| `.claude/commands/project-scan.md` |  |  |
| `.claude/commands/bug-prepare.md` |  |  |
| `ARCHITECTURE_NOTES.md` |  |  |
| Bun |  |  |
| cc-haha |  |  |
| CCR v2.0.0 |  |  |
| `start-cc-haha.sh` 或 GUI 启动器 |  |  |

</div>

---

## <center>参考链接</center>

&emsp;&emsp;本节列出本课涉及的官方文档和开源项目入口。工具版本变化较快，遇到命令或字段不一致时，优先查看这些来源，再回到本讲义中的验证方法。

- ClaudeCode 安装文档：https://code.claude.com/docs/en/setup

- ClaudeCode Settings 文档：https://code.claude.com/docs/en/settings

- ClaudeCode Skills 与 Slash Commands 文档：https://code.claude.com/docs/en/slash-commands

- ClaudeCode VS Code 扩展：https://marketplace.visualstudio.com/items?itemName=anthropic.claude-code

- cc-switch Releases：https://github.com/farion1231/cc-switch/releases

- OpenHands 仓库：https://github.com/All-Hands-AI/OpenHands

- Claw Code 仓库：https://github.com/ultraworkers/claw-code

- Claurst 仓库：https://github.com/Kuberwastaken/claurst

- cc-haha 仓库：https://github.com/NanmiCoder/cc-haha

- Claude Code Router（CCR v2.0.0）：https://github.com/musistudio/claude-code-router

- Bun 安装文档：https://bun.sh/docs/installation

- DeepSeek API 文档：https://api-docs.deepseek.com/